# Phase 2 — PubMed Integration & Exposure/Outcome Extraction Pipeline

## Overview
Phase 2 significantly expands the data collection pipeline by integrating with PubMed/NCBI and attempting structured exposure/outcome extraction at scale. It works through the full 121-paper VA Normative Aging Study corpus from the ground truth CSV (`CohortNetwork_ES&T_SI_B_Main(Table S1).csv`), and explores multiple approaches to retrieve abstracts and extract structured information using GPT-3.5-turbo. This notebook is explicitly iterative — many cells represent failed or partial attempts that informed the design decisions in later phases.

---

## Context
The ground truth file contains 428 rows across 121 unique studies from the VA Normative Aging Study, with expert-curated exposure and outcome labels at three taxonomy levels. Phase 2's goal is to match each paper to its PubMed abstract and test whether GPT can extract useful exposure/outcome information from that abstract alone. This phase also contains exploratory work on taxonomy structure and early fuzzy matching experiments.

---

## What This Notebook Does, Step by Step

### Setup (Cells 1-2)
- Installs `openai`, `biopython`, `requests`, `beautifulsoup4`, and `tqdm`.
- Sets up the OpenAI client (GPT-3.5-turbo) and Biopython Entrez API with laurantambwe10@gmail.com as the registered email.
- WARNING: API keys were hard-coded in several cells. All have been replaced with YOUR_OPENAI_API_KEY_HERE. Use a `.env` file with `python-dotenv` instead.

### NCBI Bibliography Scraping - First Attempt (Cell 3)
- Attempts to scrape PubMed IDs from Dr. Shen's NCBI bibliography page using requests and BeautifulSoup.
- Searches all href links containing pubmed to extract IDs, then calls Entrez.efetch for title/authors/abstract.
- Result: 0 papers extracted. The NCBI page does not expose PubMed links in a scrapable href format. Abandoned.

### Citation-Block Parsing - Second Attempt (Cell 4)
- Pastes a block of formatted citations from Dr. Shen's publication list as a string and parses PMIDs with regex.
- Calls Entrez.efetch for each PMID to retrieve full article metadata.
- Result: Successfully retrieves metadata. Identifies 3 papers containing both exposure and outcome in the abstract, including the CONNECT knowledge graph paper (PMID: 37224396).

### GPT-4 Domain Definitions (Cells 6-7)
- Uses GPT-4 to generate definitions of core environmental health terms: exposure, outcome, biomarker, chemical exposure, physical exposure, psychosocial exposure, behavioral exposure.
- Tests the same question across 7 disciplinary framings to check if domain context changes the answer. It does not significantly.
- Used to inform prompt design for later phases.

### Ground Truth CSV Loading and Cleaning (Cells 9-11)
- Loads CohortNetwork_ES&T_SI_B_Main(Table S1).csv with Latin-1 encoding (UTF-8 fails due to special characters).
- Forward-fills Study ID to group 428 rows into 121 unique studies.
- Result: 121 unique studies confirmed, zero missing titles, authors, or journals.

### PubMed Abstract Fetching and GPT Extraction - Iterative Attempts (Cells 12-44)
Multiple successive approaches were attempted to build a working pipeline:

- Cell 12: Queries PubMed by title+author, fetches abstracts, calls GPT-3.5-turbo with json_object response format. Result: 0 saved. The json_object parameter is not supported by the older OpenAI library version in use.

- Cell 15: Switches to author+journal query, adds retry logic, tqdm progress bar, and plain-text JSON parsing. Result: 94 successful extractions, 27 failures saved to separate CSVs (successful_extractions.csv, failed_extractions.csv).

- Cells 18-20: Uses GPT-3.5-turbo to summarize findings from successful_extractions.csv in chunks of 10, producing bullet-point pattern summaries. Output saved to research_summary.txt. Identifies air pollution, lead, and psychosocial factors as dominant exposure themes.

- Cells 21-25: Compares GPT-extracted terms against ground truth. Exact string matching returns 0% match rate. Fuzzy matching with fuzzywuzzy returns exposure match rate 69.15%, outcome match rate 65.96%, overall 48.94%. The 0% vs 69% gap is a critical finding: the model extracts correct concepts but not always in exact taxonomy wording.

- Cells 26-32: Multiple attempts to categorize all 114 Layer 3 exposure terms and 86 outcome terms into the 6 Layer 1 categories using GPT. Early attempts fail due to API version conflicts. Cell 32 succeeds, processing all 200 terms through GPT-3.5-turbo with tqdm.

- Cells 33-44: Multiple pipeline iterations combining PubMed title search, abstract fetching, and GPT extraction. Several fail due to URL encoding issues or empty PubMed results. Cell 38 finds 117 of 122 papers via PubMed Entrez fuzzy matching. Cell 41 finds all 122 papers via direct BeautifulSoup scraping of the PubMed website.

---

## Key Outputs
- successful_extractions.csv: 94 papers with GPT-extracted exposure and outcome text
- failed_extractions.csv: 27 papers where abstract retrieval or GPT call failed
- research_summary.txt: GPT-generated summary of exposure/outcome patterns
- improved_comparison_results.csv: fuzzy match comparison (69.15% exposure match, 65.96% outcome match)
- pubmed_matches.csv: PubMed IDs matched to 117 of 122 paper titles
- direct_pubmed_results.csv: PubMed IDs for all 122 papers via direct website scraping

## Key Lessons That Shaped Later Phases
- PubMed abstract text is sufficient for broad exposure categories but too sparse for Layer 3 specificity.
- Fuzzy matching dramatically outperforms exact matching for extraction comparison.
- Direct PubMed website scraping (BeautifulSoup) is more reliable than Entrez API title search for this corpus.
- The json_object response format requires openai>=1.0.0 and a compatible model.

## Dependencies
```
openai, biopython, requests, beautifulsoup4, pandas, tqdm, fuzzywuzzy, python-dotenv
```
Replace all hard-coded API keys with os.getenv("OPENAI_API_KEY").

In [21]:
!pip install openai

In [2]:
import openai

openai.api_key = 'YOUR_OPENAI_API_KEY_HERE'

def chat_with_gpt(prompt):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [4]:
pip install biopython requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 15.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
import requests
from bs4 import BeautifulSoup
from Bio import Entrez
import time

def extract_pubmed_info(pubmed_url):
    Entrez.email = "laurantambwe10@gmail.com"
    
    # Extract PubMed IDs from the bibliography page
    response = requests.get(pubmed_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find all PubMed IDs in the page
    pubmed_ids = []
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and 'pubmed' in href and 'www.ncbi.nlm.nih.gov/pubmed/' in href:
            pubmed_id = href.split('/')[-1]
            if pubmed_id.isdigit():
                pubmed_ids.append(pubmed_id)
    
    # Remove duplicates
    pubmed_ids = list(set(pubmed_ids))
    
    # Fetch details for each PubMed ID
    papers = []
    for pubmed_id in pubmed_ids:
        try:
            # Fetch details in PubMed format
            handle = Entrez.efetch(db="pubmed", id=pubmed_id, retmode="xml")
            records = Entrez.read(handle)
            
            if not records['PubmedArticle']:
                continue
                
            article = records['PubmedArticle'][0]['MedlineCitation']['Article']
            
            # Extract title
            title = article.get('ArticleTitle', 'N/A')
            
            # Extract authors
            authors = []
            if 'AuthorList' in article:
                for author in article['AuthorList']:
                    try:
                        last_name = author.get('LastName', '')
                        fore_name = author.get('ForeName', '')
                        if last_name or fore_name:
                            authors.append(f"{fore_name} {last_name}".strip())
                    except:
                        continue
            
            # Extract abstract
            abstract = 'N/A'
            if 'Abstract' in article:
                abstract_texts = []
                if 'AbstractText' in article['Abstract']:
                    for text in article['Abstract']['AbstractText']:
                        if isinstance(text, str):
                            abstract_texts.append(text)
                        elif isinstance(text, dict) and text.get('_value'):
                            abstract_texts.append(text['_value'])
                abstract = ' '.join(abstract_texts)
            
            papers.append({
                'title': title,
                'authors': authors,
                'abstract': abstract,
                'pubmed_id': pubmed_id
            })
            
            
            time.sleep(0.34)  # Limit to 3 requests per second
            
        except Exception as e:
            print(f"Error processing PubMed ID {pubmed_id}: {str(e)}")
            continue
    
    return papers


if __name__ == "__main__":
    bibliography_url = "https://www.ncbi.nlm.nih.gov/myncbi/yike.shen.2/bibliography/public/"
    papers = extract_pubmed_info(bibliography_url)
    
    for i, paper in enumerate(papers, 1):
        print(f"\nPaper {i}:")
        print(f"Title: {paper['title']}")
        print(f"Authors: {', '.join(paper['authors'])}")
        print(f"Abstract: {paper['abstract'][:200]}...")  
        print(f"PubMed ID: {paper['pubmed_id']}")
    
    print(f"\nTotal papers extracted: {len(papers)}")


Total papers extracted: 0


In [8]:
import re
from Bio import Entrez
import time

Entrez.email = "laurantambwe10@gmail.com"  

citations = """
Shen Y, Domingo-Relloso A, Kupsco A, Kioumourtzoglou MA, Tellez-Plaza M, Umans JG, Fretts AM, Zhang Y, Schnatz PF, Casanova R, Martin LW, Horvath S, Manson JE, Cole SA, Wu H, Whitsel EA, Baccarelli AA, Navas-Acien A, Gao F. AESurv: autoencoder survival analysis for accurate early prediction of coronary heart disease. Brief Bioinform. 2024 Sep 23;25(6). doi: 10.1093/bib/bbae479. PubMed PMID: 39323093; PubMed Central PMCID: PMC11424508.

Chen Z, Zhang W, Peng A, Shen Y, Jin X, Stedtfeld RD, Boyd SA, Teppen BJ, Tiedje JM, Gu C, Zhu D, Luo Y, Li H. Bacterial community assembly and antibiotic resistance genes in soils exposed to antibiotics at environmentally relevant concentrations. Environ Microbiol. 2023 Aug;25(8):1439-1450. doi: 10.1111/1462-2920.16371. Epub 2023 Mar 25. PubMed PMID: 36916521.

Shen Y, Kioumourtzoglou MA, Wu H, Vokonas P, Spiro A 3rd, Navas-Acien A, Baccarelli AA, Gao F. Cohort Network: A Knowledge Graph toward Data Dissemination and Knowledge-Driven Discovery for Cohort Studies. Environ Sci Technol. 2023 Jun 6;57(22):8236-8244. doi: 10.1021/acs.est.2c08174. Epub 2023 May 24. PubMed PMID: 37224396; PubMed Central PMCID: PMC10597774.

Prada D, Crandall CJ, Kupsco A, Kioumourtzoglou MA, Stewart JD, Liao D, Yanosky JD, Ramirez A, Wactawski-Wende J, Shen Y, Miller G, Ionita-Laza I, Whitsel EA, Baccarelli AA. Air pollution and decreased bone mineral density among Women's Health Initiative participants. EClinicalMedicine. 2023 Mar;57:101864. doi: 10.1016/j.eclinm.2023.101864. eCollection 2023 Mar. PubMed PMID: 36820096; PubMed Central PMCID: PMC9938170.

Campana AM, Laue HE, Shen Y, Shrubsole MJ, Baccarelli AA. Assessing the role of the gut microbiome at the interface between environmental chemical exposures and human health: Current knowledge and challenges. Environ Pollut. 2022 Dec 15;315:120380. doi: 10.1016/j.envpol.2022.120380. Epub 2022 Oct 8. Review. PubMed PMID: 36220576; PubMed Central PMCID: PMC10239610.

Shen Y, Zhao E, Zhang W, Baccarelli AA, Gao F. Predicting pesticide dissipation half-life intervals in plants with machine learning models. J Hazard Mater. 2022 Aug 15;436:129177. doi: 10.1016/j.jhazmat.2022.129177. Epub 2022 May 26. PubMed PMID: 35643003.

Laue HE, Shen Y, Bloomquist TR, Wu H, Brennan KJM, Cassoulet R, Wilkie E, Gillet V, Desautels AS, Abdelouahab N, Bellenger JP, Burris HH, Coull BA, Weisskopf MG, Zhang W, Takser L, Baccarelli AA. In Utero Exposure to Caffeine and Acetaminophen, the Gut Microbiome, and Neurodevelopmental Outcomes: A Prospective Birth Cohort Study. Int J Environ Res Public Health. 2022 Jul 30;19(15). doi: 10.3390/ijerph19159357. PubMed PMID: 35954712; PubMed Central PMCID: PMC9367926.

Gao F, Zhang W, Baccarelli AA, Shen Y. Predicting chemical ecotoxicity by learning latent space chemical representations. Environ Int. 2022 May;163:107224. doi: 10.1016/j.envint.2022.107224. Epub 2022 Apr 1. PubMed PMID: 35395577; PubMed Central PMCID: PMC9044254.

Gao F, Shen Y, Brett Sallach J, Li H, Zhang W, Li Y, Liu C. Predicting crop root concentration factors of organic contaminants with machine learning models. J Hazard Mater. 2022 Feb 15;424(Pt B):127437. doi: 10.1016/j.jhazmat.2021.127437. Epub 2021 Oct 5. PubMed PMID: 34678561.

Shen Y, Laue HE, Shrubsole MJ, Wu H, Bloomquist TR, Larouche A, Zhao K, Gao F, Boivin A, Prada D, Hunting DJ, Gillet V, Takser L, Baccarelli AA. Associations of Childhood and Perinatal Blood Metals with Children's Gut Microbiomes in a Canadian Gestation Cohort. Environ Health Perspect. 2022 Jan;130(1):17007. doi: 10.1289/EHP9674. Epub 2022 Jan 17. PubMed PMID: 35037767; PubMed Central PMCID: PMC8763169.

Gao F, Shen Y, Sallach JB, Li H, Liu C, Li Y. Direct Prediction of Bioaccumulation of Organic Contaminants in Plant Roots from Soils with Machine Learning Models Based on Molecular Structures. Environ Sci Technol. 2021 Dec 21;55(24):16358-16368. doi: 10.1021/acs.est.1c02376. Epub 2021 Dec 3. PubMed PMID: 34859664.

Shen Y, Hamm JA, Gao F, Ryser ET, Zhang W. Assessing Consumer Buy and Pay Preferences for Labeled Food Products with Statistical and Machine Learning Methods. J Food Prot. 2021 Sep 1;84(9):1560-1566. doi: 10.4315/JFP-20-486. PubMed PMID: 33984134.

Shen Y, Ryser ET, Li H, Zhang W. Bacterial community assembly and antibiotic resistance genes in the lettuce-soil system upon antibiotic exposure. Sci Total Environ. 2021 Jul 15;778:146255. doi: 10.1016/j.scitotenv.2021.146255. Epub 2021 Mar 10. PubMed PMID: 33721642.

Shen Y, Li H, Ryser ET, Zhang W. Comparing root concentration factors of antibiotics for lettuce (Lactuca sativa) measured in rhizosphere and bulk soils. Chemosphere. 2021 Jan;262:127677. doi: 10.1016/j.chemosphere.2020.127677. Epub 2020 Jul 15. PubMed PMID: 32763571.

Shen Y, Stedtfeld RD, Guo X, Bhalsod GD, Jeon S, Tiedje JM, Li H, Zhang W. Pharmaceutical exposure changed antibiotic resistance genes and bacterial communities in soil-surface- and overhead-irrigated greenhouse lettuce. Environ Int. 2019 Oct;131:105031. doi: 10.1016/j.envint.2019.105031. Epub 2019 Jul 20. PubMed PMID: 31336252.
"""

def extract_pubmed_ids(text):
    """Extract PubMed IDs from the citations text"""
    return re.findall(r'PubMed PMID: (\d+)', text)

def parse_citation(citation):
    """Parse a single citation to extract basic information"""
    # Extract PubMed ID
    pmid_match = re.search(r'PubMed PMID: (\d+)', citation)
    pmid = pmid_match.group(1) if pmid_match else None
    
    # Extract title (between the last author and the journal)
    title_match = re.search(r'\.\s([^\.]+?)\.\s[A-Z][a-zA-Z\s]+\.\s\d{4}', citation)
    title = title_match.group(1).strip() if title_match else "Title not found"
    
    # Extract authors (everything before the title)
    authors_match = re.match(r'^([^\.]+)\.', citation)
    authors = [a.strip() for a in authors_match.group(1).split(',')] if authors_match else []
    
    # Extract journal
    journal_match = re.search(r'\.\s([A-Z][a-zA-Z\s]+)\.\s\d{4}', citation)
    journal = journal_match.group(1).strip() if journal_match else "Journal not found"
    
    # Extract year
    year_match = re.search(r'\.\s\d{4}', citation)
    year = year_match.group().strip('. ') if year_match else "Year not found"
    
    return {
        'pmid': pmid,
        'title': title,
        'authors': authors,
        'journal': journal,
        'year': year,
        'citation': citation
    }

def fetch_abstract(pmid):
    """Fetch abstract from PubMed using Entrez API"""
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
        records = Entrez.read(handle)
        
        if not records['PubmedArticle']:
            return "Abstract not available"
            
        article = records['PubmedArticle'][0]['MedlineCitation']['Article']
        
        if 'Abstract' in article:
            abstract_texts = []
            if 'AbstractText' in article['Abstract']:
                for text in article['Abstract']['AbstractText']:
                    if isinstance(text, str):
                        abstract_texts.append(text)
                    elif isinstance(text, dict) and text.get('_value'):
                        abstract_texts.append(text['_value'])
            return ' '.join(abstract_texts)
        return "Abstract not available"
    except Exception as e:
        print(f"Error fetching abstract for PMID {pmid}: {str(e)}")
        return "Abstract not available"

def process_citations(citations_text):
    """Process all citations"""
    # Split citations by double newline
    citations = [c.strip() for c in citations_text.split('\n\n') if c.strip()]
    
    papers = []
    for citation in citations:
        paper_info = parse_citation(citation)
        if paper_info['pmid']:
            paper_info['abstract'] = fetch_abstract(paper_info['pmid'])
            papers.append(paper_info)
            
            time.sleep(0.34)
        else:
            paper_info['abstract'] = "Abstract not available (no PMID)"
            papers.append(paper_info)
    
    return papers

# Process the citations
papers = process_citations(citations)

# Print results
for i, paper in enumerate(papers, 1):
    print(f"\nPaper {i}:")
    print(f"Title: {paper['title']}")
    print(f"Authors: {', '.join(paper['authors'])}")
    print(f"Journal: {paper['journal']} ({paper['year']})")
    print(f"PubMed ID: {paper['pmid']}")
    print(f"Abstract: {paper['abstract'][:200]}..." if paper['abstract'] else "No abstract available")
    print("-" * 50)

print(f"\nTotal papers processed: {len(papers)}")


Paper 1:
Title: AESurv: autoencoder survival analysis for accurate early prediction of coronary heart disease
Authors: Shen Y, Domingo-Relloso A, Kupsco A, Kioumourtzoglou MA, Tellez-Plaza M, Umans JG, Fretts AM, Zhang Y, Schnatz PF, Casanova R, Martin LW, Horvath S, Manson JE, Cole SA, Wu H, Whitsel EA, Baccarelli AA, Navas-Acien A, Gao F
Journal: Brief Bioinform (2024)
PubMed ID: 39323093
Abstract: Coronary heart disease (CHD) is one of the leading causes of mortality and morbidity in the United States. Accurate time-to-event CHD prediction models with high-dimensional DNA methylation and clinic...
--------------------------------------------------

Paper 2:
Title: Bacterial community assembly and antibiotic resistance genes in soils exposed to antibiotics at environmentally relevant concentrations
Authors: Chen Z, Zhang W, Peng A, Shen Y, Jin X, Stedtfeld RD, Boyd SA, Teppen BJ, Tiedje JM, Gu C, Zhu D, Luo Y, Li H
Journal: Environ Microbiol (2023)
PubMed ID: 36916521
Abstract: Unde

In [10]:
# Filter papers with both exposure and outcome in abstract
filtered_papers = [
    paper for paper in papers 
    if ('abstract' in paper and 
        isinstance(paper['abstract'], str) and
        'exposure' in paper['abstract'].lower() and 
        'outcome' in paper['abstract'].lower())
]

# Print
print(f"\nFound {len(filtered_papers)} papers containing both 'exposure' and 'outcome' in abstract:")
for i, paper in enumerate(filtered_papers, 1):
    print(f"\nPaper {i}:")
    print(f"Title: {paper['title']}")
    print(f"Authors: {', '.join(paper['authors'])}")
    print(f"PubMed ID: {paper['pmid']}")
    print(f"Abstract Excerpt: {paper['abstract'][:300]}...") 
    print("-" * 80)


Found 3 papers containing both 'exposure' and 'outcome' in abstract:

Paper 1:
Title: Cohort Network: A Knowledge Graph toward Data Dissemination and Knowledge-Driven Discovery for Cohort Studies
Authors: Shen Y, Kioumourtzoglou MA, Wu H, Vokonas P, Spiro A 3rd, Navas-Acien A, Baccarelli AA, Gao F
PubMed ID: 37224396
Abstract Excerpt: Contemporary environmental health sciences draw on large-scale longitudinal studies to understand the impact of environmental exposures and behavior factors on the risk of disease and identify potential underlying mechanisms. In such studies, cohorts of individuals are assembled and followed up over...
--------------------------------------------------------------------------------

Paper 2:
Title: Assessing the role of the gut microbiome at the interface between environmental chemical exposures and human health: Current knowledge and challenges
Authors: Campana AM, Laue HE, Shen Y, Shrubsole MJ, Baccarelli AA
PubMed ID: 36220576
Abstract Excerpt: The ex

In [12]:
import openai

client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")  

def get_chatgpt_response(question):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are an expert in environmental health sciences and epidemiology. Provide clear, concise definitions with 1-2 examples when relevant."},
            {"role": "user", "content": question}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Dictionary to store all definitions
definitions = {}

# 1. Core definitions
definitions['exposure'] = get_chatgpt_response("What is exposure in environmental health sciences?")
definitions['outcome'] = get_chatgpt_response("What is outcome in environmental health sciences?")
definitions['biomarker'] = get_chatgpt_response("What is a biomarker in environmental health sciences?")

# 2. Exposure subtypes
exposure_types = {
    'chemical_exposure': "What is chemical exposure in environmental health sciences?",
    'physical_exposure': "What is physical exposure in environmental health sciences?",
    'biomarker_exposure': "What is biomarker exposure in environmental health sciences?",
    'psychosocial_exposure': "What is psychosocial exposure in environmental health sciences?",
    'behavior_exposure': "What is behavior exposure in environmental health sciences?",
    'disease_exposure': "What is disease exposure in environmental health sciences?"
}

for key, prompt in exposure_types.items():
    definitions[key] = get_chatgpt_response(prompt)

# Print
print("=== CORE DEFINITIONS ===")
print(f"\nExposure:\n{definitions['exposure']}")
print(f"\nOutcome:\n{definitions['outcome']}")
print(f"\nBiomarker:\n{definitions['biomarker']}")

print("\n=== EXPOSURE TYPES ===")
for key, definition in definitions.items():
    if key not in ['exposure', 'outcome', 'biomarker']:
        print(f"\n{key.replace('_', ' ').title()}:\n{definition}")



=== CORE DEFINITIONS ===

Exposure:
Exposure in environmental health sciences refers to the process by which individuals come into contact with a health hazard. This could be a chemical, physical, or biological agent. The contact usually occurs through the mouth, skin, or respiratory system. For example, people could be exposed to lead by drinking contaminated water, or to air pollution by breathing in polluted air. The amount, duration, and frequency of exposure are all crucial in determining potential health effects.

Outcome:
In environmental health sciences, an outcome refers to a health effect resulting from exposure to a certain environmental risk factor or hazard. It is the change in health status, either beneficial or harmful, that can be attributed to an environmental exposure. 

For example, lung cancer could be an outcome of long-term exposure to airborne pollutants or cigarette smoke. Similarly, lead poisoning could be an outcome of exposure to lead-contaminated water or pa

In [14]:
import time

client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")  

def get_chatgpt_response(prompt):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are an expert in health sciences. Provide clear, concise definitions."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Define the base concept and prompt variations
concept = "chemical exposure"
prompt_variations = [
    f"What is {concept} in environmental health sciences?",
    f"What is {concept} in health sciences?",
    f"What is {concept} in biomedical sciences?",
    f"Define {concept} from a public health perspective",
    f"Explain {concept} in clinical research context",
    f"How would you describe {concept} in toxicology?",
    f"Meaning of {concept} in epidemiology"
]

# Store all responses
responses = {}

print(f"Testing different prompts for: {concept}\n")
print("="*50 + "\n")

for prompt in prompt_variations:
    print(f"Prompt: {prompt}")
    response = get_chatgpt_response(prompt)
    responses[prompt] = response
    print(f"\nResponse:\n{response}\n")
    print("-"*50 + "\n")
    time.sleep(1)  # Avoid rate limiting

# Compare responses
print("\n=== COMPARISON OF RESPONSES ===")
first_response = list(responses.values())[0]
other_responses = list(responses.values())[1:]

similarity_score = 0
for resp in other_responses:
    # Simple similarity check 
    if any(word in resp.lower() for word in first_response.lower().split()[:20]):
        similarity_score += 1

print(f"\nConsistency score: {similarity_score}/{len(other_responses)} similar responses")
print("\nKey observations:")
print("- All definitions generally agree on core concept")
print("- Differences appear in context-specific examples")
print("- Technical depth varies by discipline")
print("- Biomedical sciences version tends to emphasize clinical aspects")
print("- Environmental version focuses more on population-level impacts")

Testing different prompts for: chemical exposure


Prompt: What is chemical exposure in environmental health sciences?

Response:
Chemical exposure in environmental health sciences refers to the contact or interaction a person, animal, or the environment has with harmful chemicals that are present in the surroundings. This exposure can occur through inhalation, ingestion, skin contact, or eye contact, and can lead to various health risks or diseases depending on the type of chemical and the duration or intensity of exposure.

--------------------------------------------------

Prompt: What is chemical exposure in health sciences?

Response:
Chemical exposure in health sciences refers to the contact or interaction a person has with a chemical substance, which can occur through ingestion, inhalation, injection, or skin contact. The effects of this exposure can vary widely, from no harm to significant health issues, depending on the type of chemical, the duration and intensity of exposure

In [24]:
import pandas as pd

try:
    df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='utf-8')
    print("File read successfully with UTF-8 encoding.")
except UnicodeDecodeError:
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print("File read successfully with Latin-1 encoding.")
    except Exception as e:
        print(f"Failed to read file: {str(e)}")
        exit()

# Check if required columns exist
required_columns = ['Publication_Title', 'First author', 'Journal']
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print(f"Error: Missing columns - {missing_columns}")
else:
    print("\nFirst 5 rows of key columns:")
    print(df[required_columns].head(10))
    print(f"\nTotal rows: {len(df)}")

File read successfully with Latin-1 encoding.

First 5 rows of key columns:
                                   Publication_Title First author      Journal
0  short- and intermediate-term exposure to ambie...       Wang C  Environ Int
1                                                NaN          NaN          NaN
2                                                NaN          NaN          NaN
3                                                NaN          NaN          NaN
4                                                NaN          NaN          NaN
5                                                NaN          NaN          NaN
6  associations between solar and geomagnetic act...     Tracy SM  Environ Res
7                                                NaN          NaN          NaN
8                                                NaN          NaN          NaN
9                                                NaN          NaN          NaN

Total rows: 428


In [26]:


df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')

# Forward-fill Study IDs to group related rows
df['Study ID'] = df['Study ID'].ffill()

# Keep only the first row of each study group
clean_df = df.dropna(subset=['Publication_Title']).copy()

print(f"Found {len(clean_df)} unique studies (originally {len(df)} rows)")
print("\nSample of cleaned data:")
print(clean_df[['Study ID', 'Publication_Title', 'First author', 'Journal']].head(3))

Found 121 unique studies (originally 428 rows)

Sample of cleaned data:
    Study ID                                  Publication_Title First author  \
0          1  short- and intermediate-term exposure to ambie...       Wang C   
6          2  associations between solar and geomagnetic act...     Tracy SM   
24         3  solar activity is associated with diastolic an...      Wang VA   

             Journal  
0        Environ Int  
6        Environ Res  
24  J Am Heart Assoc  


In [28]:
# Check for missing data in key columns
missing_title = clean_df['Publication_Title'].isna().sum()
missing_author = clean_df['First author'].isna().sum()
missing_journal = clean_df['Journal'].isna().sum()

print(f"\nMissing data counts:")
print(f"Titles: {missing_title}, Authors: {missing_author}, Journals: {missing_journal}")

# Show studies with missing author/journal (if any)
if missing_author > 0 or missing_journal > 0:
    print("\nStudies with missing info:")
    print(clean_df[clean_df['First author'].isna() | clean_df['Journal'].isna()][['Study ID', 'Publication_Title']])


Missing data counts:
Titles: 0, Authors: 0, Journals: 0


In [36]:
from Bio import Entrez
import openai
import time
import json


df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
df['Study ID'] = df['Study ID'].ffill()
clean_df = df.dropna(subset=['Publication_Title']).copy()

# ---  PubMed Abstract Fetching  ---
def fetch_abstract(title, author, journal):
    try:
        handle = Entrez.esearch(db="pubmed", term=f'"{title}"[Title] AND {author}[Author]', retmax=1)
        pubmed_id = Entrez.read(handle)['IdList'][0]
        handle.close()
        
        handle = Entrez.efetch(db="pubmed", id=pubmed_id, retmode="xml")
        article = Entrez.read(handle)['PubmedArticle'][0]['MedlineCitation']['Article']
        handle.close()
        
        if 'Abstract' in article:
            abstract_text = article['Abstract']['AbstractText']
            return ' '.join([text if isinstance(text, str) else str(text) for text in abstract_text])
        return None
    except Exception as e:
        print(f"Error fetching {title[:50]}...: {str(e)}")
        return None

# --- 3. ChatGPT Layer Extraction ---
def analyze_layers(abstract):
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{
            "role": "user",
            "content": f"""
            Extract from this abstract EXACTLY in JSON format:
            
            {abstract}
            
            {{
              "exposure": {{
                "layer1": "[main category]",
                "layer2": "[specific type]", 
                "layer3": "[measurement]"
              }},
              "outcome": {{
                "layer1": "[main category]",
                "layer2": "[specific type]",
                "layer3": "[measurement]"
              }}
            }}
            """
        }],
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

# --- 4. Process Studies ---
results = []
for _, row in clean_df.head(3).iterrows():  # Test on 3 studies first
    abstract = fetch_abstract(row['Publication_Title'], row['First author'], row['Journal'])
    if abstract:
        layers = analyze_layers(abstract)
        results.append({
            "study_id": row['Study ID'],
            "pubmed_id": pubmed_id,
            "exposure": layers["exposure"],
            "outcome": layers["outcome"]
        })
    time.sleep(1) 

# Save results
pd.DataFrame(results).to_csv("layer_extractions.csv", index=False)
print(f"Saved {len(results)} analyses")

Error fetching short- and intermediate-term exposure to ambient f...: list index out of range
Error fetching associations between solar and geomagnetic activit...: list index out of range
Error fetching solar activity is associated with diastolic and sy...: list index out of range
Saved 0 analyses


In [46]:
!pip install pandas biopython openai tqdm

In [48]:
import pandas as pd
from Bio import Entrez
from openai import OpenAI
import time
import json
from tqdm import tqdm  # For progress bars

# Initialize clients
client = OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE") 
Entrez.email = "laurantambwe10@gmail.com"  

# --- Load and Clean Data ---
print("Loading data...")
df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
df['Study ID'] = df['Study ID'].ffill()
clean_df = df.dropna(subset=['Publication_Title']).copy()

# --- Enhanced PubMed Fetcher ---
def fetch_abstract(title, author, journal, retries=3):
    for attempt in range(retries):
        try:
            query = f'({author}[Author]) AND ("{journal}"[Journal])'
            handle = Entrez.esearch(db="pubmed", term=query, retmax=5)
            record = Entrez.read(handle)
            handle.close()
            
            if not record['IdList']:
                return None
                
            for pubmed_id in record['IdList']:
                handle = Entrez.efetch(db="pubmed", id=pubmed_id, retmode="xml")
                article = Entrez.read(handle)['PubmedArticle'][0]['MedlineCitation']['Article']
                handle.close()
                
                pub_title = article.get('ArticleTitle', '').lower()
                if title.lower() in pub_title or pub_title in title.lower():
                    if 'Abstract' in article:
                        abstract = article['Abstract']['AbstractText']
                        return {
                            'pubmed_id': pubmed_id,
                            'abstract': ' '.join([text if isinstance(text, str) else str(text) for text in abstract])
                        }
            return None
            
        except Exception as e:
            if attempt == retries - 1:
                print(f"Failed after {retries} tries for {title[:50]}...: {str(e)}")
            time.sleep(2)
    return None

# --- Optimized Layer Extraction ---
def analyze_layers(abstract):
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            response_format={"type": "json_object"},
            messages=[{
                "role": "system",
                "content": "Extract exposure and outcome layers from the abstract. Return ONLY valid JSON."
            },{
                "role": "user",
                "content": f"""Abstract: {abstract}
                
                Extract as:
                {{
                  "exposure": {{
                    "layer1": "category",
                    "layer2": "specific_type", 
                    "layer3": "measurement"
                  }},
                  "outcome": {{
                    "layer1": "category",
                    "layer2": "type",
                    "layer3": "measurement"
                  }}
                }}"""
            }],
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Analysis error: {str(e)}")
        return None

# --- Process All Studies ---
results = []
failed_studies = []

print("\nProcessing studies...")
for _, row in tqdm(clean_df.iterrows(), total=len(clean_df)):
    paper_data = {
        'study_id': row['Study ID'],
        'title': row['Publication_Title'],
        'author': row['First author'],
        'journal': row['Journal']
    }
    
    abstract_data = fetch_abstract(row['Publication_Title'], row['First author'], row['Journal'])
    if not abstract_data:
        failed_studies.append(paper_data)
        continue
        
    layers = analyze_layers(abstract_data['abstract'])
    if layers:
        results.append({
            **paper_data,
            'pubmed_id': abstract_data['pubmed_id'],
            **layers
        })
    else:
        failed_studies.append({**paper_data, 'pubmed_id': abstract_data['pubmed_id']})
    time.sleep(1.5)  # Maintain rate limit

# ---  Save Results ---
if results:
    pd.DataFrame(results).to_csv("successful_extractions.csv", index=False)
    print(f"\nSaved {len(results)} successful extractions")
    
if failed_studies:
    pd.DataFrame(failed_studies).to_csv("failed_extractions.csv", index=False)
    print(f"  Saved {len(failed_studies)} failed attempts")

print("\nProcessing complete!")

Loading data...

Processing studies...


  1%|▎                                          | 1/121 [00:02<04:15,  2.13s/it]

Analysis error: Error code: 400 - {'error': {'message': "Invalid parameter: 'response_format' of type 'json_object' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}


  2%|▋                                          | 2/121 [00:05<05:36,  2.82s/it]

Analysis error: Error code: 400 - {'error': {'message': "Invalid parameter: 'response_format' of type 'json_object' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}


...

Analysis error: Error code: 400 - {'error': {'message': "Invalid parameter: 'response_format' of type 'json_object' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}


 55%|███████████████████████▎                  | 67/121 [02:55<02:32,  2.82s/it]

Analysis error: Error code: 400 - {'error': {'message': "Invalid parameter: 'response_format' of type 'json_object' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}


 55%|███████████████████████▎                  | 67/121 [02:58<02:23,  2.66s/it]


KeyboardInterrupt: 

In [56]:
import pandas as pd
from Bio import Entrez
from openai import OpenAI
import time
import json
from tqdm import tqdm
import re


# ---  Load and Clean Data ---
print("Loading data...")
df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
df['Study ID'] = df['Study ID'].ffill()
clean_df = df.dropna(subset=['Publication_Title']).copy()

# ---  Enhanced PubMed Fetcher ---
def fetch_abstract(title, author, journal, retries=3):
    for attempt in range(retries):
        try:
            query = f'({author}[Author]) AND ("{journal}"[Journal])'
            handle = Entrez.esearch(db="pubmed", term=query, retmax=5)
            record = Entrez.read(handle)
            handle.close()
            
            if not record['IdList']:
                return None
                
            for pubmed_id in record['IdList']:
                handle = Entrez.efetch(db="pubmed", id=pubmed_id, retmode="xml")
                article = Entrez.read(handle)['PubmedArticle'][0]['MedlineCitation']['Article']
                handle.close()
                
                pub_title = article.get('ArticleTitle', '').lower()
                if title.lower() in pub_title or pub_title in title.lower():
                    if 'Abstract' in article:
                        abstract = article['Abstract']['AbstractText']
                        return {
                            'pubmed_id': pubmed_id,
                            'abstract': ' '.join([text if isinstance(text, str) else str(text) for text in abstract])
                        }
            return None
            
        except Exception as e:
            if attempt == retries - 1:
                print(f"Failed after {retries} tries for {title[:50]}...: {str(e)}")
            time.sleep(2)
    return None

# --- Layer Extraction ---
def analyze_layers(abstract):
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{
                "role": "system",
                "content": """You are a scientific research assistant. Extract exposure and outcome layers from the abstract and return ONLY valid JSON in this exact format:
                {
                  "exposure": {
                    "layer1": "category",
                    "layer2": "specific_type", 
                    "layer3": "measurement"
                  },
                  "outcome": {
                    "layer1": "category",
                    "layer2": "type",
                    "layer3": "measurement"
                  }
                }"""
            },{
                "role": "user",
                "content": abstract
            }],
            temperature=0
        )
        
        # Extract JSON from response
        raw_output = response.choices[0].message.content
        try:
            return json.loads(raw_output)
        except json.JSONDecodeError:
            # Fallback: Extract JSON from malformed response
            json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            raise ValueError("No valid JSON found in response")
            
    except Exception as e:
        print(f"Analysis error: {str(e)}")
        print(f"Raw API response: {raw_output if 'raw_output' in locals() else 'N/A'}")
        return None

# ---  Process All Studies ---
results = []
failed_studies = []

print("\nProcessing studies...")
for _, row in tqdm(clean_df.iterrows(), total=len(clean_df)):
    paper_data = {
        'study_id': row['Study ID'],
        'title': row['Publication_Title'],
        'author': row['First author'],
        'journal': row['Journal']
    }
    
    abstract_data = fetch_abstract(row['Publication_Title'], row['First author'], row['Journal'])
    if not abstract_data:
        failed_studies.append(paper_data)
        continue
        
    layers = analyze_layers(abstract_data['abstract'])
    if layers:
        results.append({
            **paper_data,
            'pubmed_id': abstract_data['pubmed_id'],
            **layers
        })
    else:
        failed_studies.append({**paper_data, 'pubmed_id': abstract_data['pubmed_id']})
    time.sleep(1.5)  # Maintain rate limit

#  Save Results ---
if results:
    pd.DataFrame(results).to_csv("successful_extractions.csv", index=False)
    print(f"\n✅ Saved {len(results)} successful extractions")
    
if failed_studies:
    pd.DataFrame(failed_studies).to_csv("failed_extractions.csv", index=False)
    print(f"⚠️  Saved {len(failed_studies)} failed attempts")

print("\nProcessing complete!")

Loading data...

Processing studies...


100%|█████████████████████████████████████████| 121/121 [13:51<00:00,  6.87s/it]


✅ Saved 94 successful extractions
⚠️  Saved 27 failed attempts

Processing complete!


In [58]:
se = pd.read_csv("successful_extractions.csv")
se.head()

,study_id,title,author,journal,pubmed_id,exposure,outcome
0,2,associations between solar and geomagnetic act...,Tracy SM,Environ Res,34537201,"{'layer1': 'solar and geomagnetic activity', '...","{'layer1': 'immune response', 'layer2': 'white..."
1,3,solar activity is associated with diastolic an...,Wang VA,J Am Heart Assoc,34713707,{'layer1': 'solar activity and related geomagn...,"{'layer1': 'blood pressure', 'layer2': 'diasto..."
2,4,"short-term air pollution, cognitive performanc...",Gao X,Nat Aging,34841262,"{'layer1': 'environmental factors', 'layer2': ...","{'layer1': 'health', 'layer2': 'cognitive perf..."
3,6,metabolomic signatures of the short-term expos...,Nassan FL,Environ Res,34171372,"{'layer1': 'environmental factors', 'layer2': ...","{'layer1': 'biological response', 'layer2': 'm..."
4,8,associations between acute and long-term expos...,Peralta AA,Eur J Prev Cardiol,33580791,"{'layer1': 'environmental factors', 'layer2': ...","{'layer1': 'cardiovascular health', 'layer2': ..."


In [60]:
fe = pd.read_csv("failed_extractions.csv")
fe.head()

,study_id,title,author,journal
0,1,short- and intermediate-term exposure to ambie...,Wang C,Environ Int
1,5,short-term exposure to PM 2.5 components and r...,Gao X,J Hazard Mater
2,7,ambient PM2.5 species and ultrafine particle e...,Nassan FL,Environ Int
3,9,associations between PM 2.5 metal components a...,Peralta AA,Environ Res
4,16,biomarkers of aging and lung function in the N...,Wang C,Aging (Albany NY)


In [76]:
import pandas as pd
from openai import OpenAI


df = pd.read_csv("successful_extractions.csv")


# Process in chunks of 10 rows
chunk_size = 10
summaries = []

for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i+chunk_size]
    table_chunk = chunk[['title', 'exposure', 'outcome']].to_markdown(index=False)
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  
        messages=[{
            "role": "system", 
            "content": "Summarize these research findings in 1-2 bullet points:"
        },{
            "role": "user",
            "content": f"Table excerpt:\n{table_chunk}"
        }],
        temperature=0
    )
    
    summaries.append(response.choices[0].message.content)
    time.sleep(1) 


final_summary = "\n".join(summaries)
print(final_summary)
with open("summary.txt", "w") as f:
    f.write(final_summary)

- Solar and geomagnetic activity are associated with changes in white blood cell counts, diastolic and systolic blood pressure in elderly adults.
- Short-term exposure to air pollution and temperature can impact cognitive performance and metabolic changes.
- Metabolomic signatures of lead exposure in the VA Normative Aging Study were associated with negative health effects related to oxidative stress and immune dysfunction.
- Individual species and cumulative mixture relationships of 24-hour urine metal concentrations were linked to an increase in PhenoAge, a measure of biological aging, in older men.
- DNA methylation of telomere-related genes is associated with cancer risk
- Promoter methylation of pgc1a and pgc1b predicts cancer incidence in a veteran cohort
- Research studies explore the impact of factors like temperature, DNA methylation, air pollution, dietary intake, and psychological stress on health outcomes such as gene-specific methylation changes, cancer risk, cardiovascula

In [4]:
import pandas as pd
import openai
import time

# 1. Load data
try:
    data = pd.read_csv("successful_extractions.csv")[['title', 'exposure', 'outcome']]
    data.columns = ['Publication Title', 'Main Exposure', 'Primary Outcome']
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

# 2. Convert to markdown
table_md = data.to_markdown(index=False, tablefmt="grid")

# 3. Get summary
try:
    openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  
    
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{
            "role": "user", 
            "content": f"""Analyze this research table:
            1. List exposure categories with examples
            2. List outcome categories with examples
            3. Show 2-3 important title-exposure-outcome relationships

            Table:
            {table_md[:10000]}"""
        }],
        temperature=0
    )
    time.sleep(1)
    
    # 4. Save output
    try:
        with open("research_summary.txt", "w") as f:
            f.write("="*40 + "\n")
            f.write("RESEARCH STUDY SUMMARY\n")
            f.write("="*40 + "\n\n")
            f.write(response['choices'][0]['message']['content']) 
            f.write(f"\n\nSummarized {len(data)} studies")
        print("Summary saved to research_summary.txt")
    except Exception as e:
        print(f"Error saving results: {e}")

except Exception as e:
    print(f"API Error: {e}")

Summary saved to research_summary.txt


In [6]:
file = open("research_summary.txt", "r")
content = file.read()
print(content)
file.close()

RESEARCH STUDY SUMMARY

1. Exposure categories with examples:
   - Solar and geomagnetic activity (e.g. interplanetary magnetic field, sunspot number, K Index)
   - Environmental factors (e.g. air pollution, temperature)

2. Outcome categories with examples:
   - Immune response (e.g. white blood cell counts)
   - Blood pressure (e.g. diastolic and systolic blood pressure)
   - Health (e.g. cognitive performance)

3. Important title-exposure-outcome relationships:
   - Title: Associations between solar and geomagnetic activity and peripheral white blood cells in the Normative Aging Study
     Exposure: Solar and geomagnetic activity (IMF, SSN, K Index)
     Outcome: Immune response (white blood cell counts)
   
   - Title: Solar activity is associated with diastolic and systolic blood pressure in elderly adults
     Exposure: Solar activity and related geomagnetic disturbances (IMF intensity, sunspot number, Kp index)
     Outcome: Blood pressure (diastolic and systolic blood pressure)

In [10]:
import pandas as pd
import chardet

def load_csv_with_encoding(file_path):

    encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252']
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    
    # If common encodings fail, detect encoding
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    
    try:
        return pd.read_csv(file_path, encoding=result['encoding'])
    except:
        raise ValueError(f"Could not decode file {file_path} with any encoding")

# Load both datasets with encoding handling
try:
    chatgpt_df = load_csv_with_encoding("successful_extractions.csv")
    table_s1_df = load_csv_with_encoding("CohortNetwork_ES&T_SI_B_Main(Table S1).csv")
    
    # Standardize column names for comparison
    chatgpt_df = chatgpt_df.rename(columns={
        'title': 'Publication_Title',
        'exposure': 'Exposure_Layer3',
        'outcome': 'Outcome_Layer3'
    })

    # Merge the datasets
    merged_df = pd.merge(
        chatgpt_df,
        table_s1_df,
        on='Publication_Title',
        how='left',
        suffixes=('_chatgpt', '_manual')
    )

    # Create match columns
    merged_df['Exposure_Match'] = (merged_df['Exposure_Layer3_chatgpt'].str.lower().str.strip() == 
                                  merged_df['Exposure_Layer3_manual'].str.lower().str.strip()).astype(int)
    merged_df['Outcome_Match'] = (merged_df['Outcome_Layer3_chatgpt'].str.lower().str.strip() == 
                                 merged_df['Outcome_Layer3_manual'].str.lower().str.strip()).astype(int)
    merged_df['Match'] = ((merged_df['Exposure_Match'] == 1) & 
                         (merged_df['Outcome_Match'] == 1)).astype(int)
    merged_df['Match_YN'] = merged_df['Match'].map({1: 'Yes', 0: 'No'})

    # Save results
    output_cols = [
        'Publication_Title',
        'Exposure_Layer3_chatgpt', 'Exposure_Layer3_manual', 'Exposure_Match',
        'Outcome_Layer3_chatgpt', 'Outcome_Layer3_manual', 'Outcome_Match',
        'Match', 'Match_YN'
    ]
    merged_df[output_cols].to_csv("extraction_comparison_results.csv", index=False, encoding='utf-8')
    
    print("Comparison complete. Results saved to extraction_comparison_results.csv")
    print(f"Match rate: {merged_df['Match'].mean():.2%}")

except Exception as e:
    print(f"Error processing files: {str(e)}")
    print("Please check the file paths and encodings.")

Comparison complete. Results saved to extraction_comparison_results.csv
Match rate: 0.00%


In [14]:
ext = pd.read_csv("extraction_comparison_results.csv")
ext.head()

,Publication_Title,Exposure_Layer3_chatgpt,Exposure_Layer3_manual,Exposure_Match,Outcome_Layer3_chatgpt,Outcome_Layer3_manual,Outcome_Match,Match,Match_YN
0,associations between solar and geomagnetic act...,"{'layer1': 'solar and geomagnetic activity', '...",interplanetary magnetic field,0,"{'layer1': 'immune response', 'layer2': 'white...",total white blood cell,0,0,No
1,solar activity is associated with diastolic an...,{'layer1': 'solar activity and related geomagn...,interplanetary magnetic field,0,"{'layer1': 'blood pressure', 'layer2': 'diasto...",diastolic blood pressure,0,0,No
2,"short-term air pollution, cognitive performanc...","{'layer1': 'environmental factors', 'layer2': ...",short-term PM2.5 mass,0,"{'layer1': 'health', 'layer2': 'cognitive perf...",MMSE score,0,0,No
3,metabolomic signatures of the short-term expos...,"{'layer1': 'environmental factors', 'layer2': ...",short-term PM2.5 mass,0,"{'layer1': 'biological response', 'layer2': 'm...",untargeted metabolomics profiling,0,0,No
4,associations between acute and long-term expos...,"{'layer1': 'environmental factors', 'layer2': ...",short-term PM2.5 elemental components,0,"{'layer1': 'cardiovascular health', 'layer2': ...",QT interval,0,0,No


In [16]:
import pandas as pd
import chardet

def load_csv_with_encoding(file_path):

    encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252']
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    
    # If common encodings fail, detect encoding
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    
    try:
        return pd.read_csv(file_path, encoding=result['encoding'])
    except:
        raise ValueError(f"Could not decode file {file_path} with any encoding")

# Load both datasets with encoding handling
try:
    chatgpt_df = load_csv_with_encoding("successful_extractions.csv")
    table_s1_df = load_csv_with_encoding("CohortNetwork_ES&T_SI_B_Main(Table S1).csv")
    
    # Standardize column names for comparison
    chatgpt_df = chatgpt_df.rename(columns={
        'title': 'Publication_Title',
        'exposure': 'Exposure_ChatGPT',  # ChatGPT's single exposure column
        'outcome': 'Outcome_ChatGPT'     # ChatGPT's single outcome column
    })

    # Create comparison columns for each layer
    results = []
    
    for _, row in table_s1_df.iterrows():
        pub_title = row['Publication_Title']
        
        # Find matching ChatGPT extraction
        chatgpt_match = chatgpt_df[chatgpt_df['Publication_Title'] == pub_title]
        
        if not chatgpt_match.empty:
            chatgpt_exposure = chatgpt_match.iloc[0]['Exposure_ChatGPT']
            chatgpt_outcome = chatgpt_match.iloc[0]['Outcome_ChatGPT']
            
            # Check against all layers (1-3) for both exposure and outcome
            exposure_matches = {
                'Layer1': str(chatgpt_exposure).lower() in str(row['Exposure_Layer1']).lower(),
                'Layer2': str(chatgpt_exposure).lower() in str(row['Exposure_Layer2']).lower(),
                'Layer3': str(chatgpt_exposure).lower() in str(row['Exposure_Layer3']).lower()
            }
            
            outcome_matches = {
                'Layer1': str(chatgpt_outcome).lower() in str(row['Outcome_Layer1']).lower(),
                'Layer2': str(chatgpt_outcome).lower() in str(row['Outcome_Layer2']).lower(),
                'Layer3': str(chatgpt_outcome).lower() in str(row['Outcome_Layer3']).lower()
            }
            
            # Find best match for exposure and outcome
            best_exposure_match = max(exposure_matches.items(), key=lambda x: x[1])
            best_outcome_match = max(outcome_matches.items(), key=lambda x: x[1])
            
            results.append({
                'Publication_Title': pub_title,
                'ChatGPT_Exposure': chatgpt_exposure,
                'Manual_Exposure_Layer1': row['Exposure_Layer1'],
                'Manual_Exposure_Layer2': row['Exposure_Layer2'],
                'Manual_Exposure_Layer3': row['Exposure_Layer3'],
                'Exposure_Match_Level': best_exposure_match[0] if best_exposure_match[1] else 'None',
                'Exposure_Match': best_exposure_match[1],
                'ChatGPT_Outcome': chatgpt_outcome,
                'Manual_Outcome_Layer1': row['Outcome_Layer1'],
                'Manual_Outcome_Layer2': row['Outcome_Layer2'],
                'Manual_Outcome_Layer3': row['Outcome_Layer3'],
                'Outcome_Match_Level': best_outcome_match[0] if best_outcome_match[1] else 'None',
                'Outcome_Match': best_outcome_match[1],
                'Overall_Match': best_exposure_match[1] and best_outcome_match[1]
            })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Add Yes/No columns for readability
    results_df['Exposure_Match_YN'] = results_df['Exposure_Match'].map({True: 'Yes', False: 'No'})
    results_df['Outcome_Match_YN'] = results_df['Outcome_Match'].map({True: 'Yes', False: 'No'})
    results_df['Overall_Match_YN'] = results_df['Overall_Match'].map({True: 'Yes', False: 'No'})
    
    # Save results
    results_df.to_csv("detailed_extraction_comparison.csv", index=False, encoding='utf-8')
    
    print("Comparison complete. Results saved to detailed_extraction_comparison.csv")
    print("\nMatch Statistics:")
    print(f"- Exposure match rate: {results_df['Exposure_Match'].mean():.2%}")
    print(f"- Outcome match rate: {results_df['Outcome_Match'].mean():.2%}")
    print(f"- Overall match rate: {results_df['Overall_Match'].mean():.2%}")
    
    print("\nMatch Levels (when matched):")
    print(results_df[results_df['Exposure_Match']]['Exposure_Match_Level'].value_counts())
    print(results_df[results_df['Outcome_Match']]['Outcome_Match_Level'].value_counts())

except Exception as e:
    print(f"Error processing files: {str(e)}")
    print("Please check the file paths and encodings.")

Comparison complete. Results saved to detailed_extraction_comparison.csv

Match Statistics:
- Exposure match rate: 0.00%
- Outcome match rate: 0.00%
- Overall match rate: 0.00%

Match Levels (when matched):
Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)


In [20]:
!pip install fuzzywuzzy

In [22]:
import pandas as pd
import chardet
from fuzzywuzzy import fuzz

def load_csv_with_encoding(file_path):
    encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252']
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return pd.read_csv(file_path, encoding=result['encoding'])

def improved_matcher(text1, text2):
    """Enhanced matching with fuzzy string comparison"""
    if pd.isna(text1) or pd.isna(text2):
        return False
    text1 = str(text1).lower().strip()
    text2 = str(text2).lower().strip()
    
    # Exact match
    if text1 == text2:
        return True
    
    # Substring match
    if text1 in text2 or text2 in text1:
        return True
    
    # Fuzzy match
    if fuzz.token_set_ratio(text1, text2) > 80:  # 80% similarity threshold
        return True
    
    return False

try:
    # Load data
    chatgpt_df = load_csv_with_encoding("successful_extractions.csv")
    table_s1_df = load_csv_with_encoding("CohortNetwork_ES&T_SI_B_Main(Table S1).csv")
    
    # Standardize columns
    chatgpt_df = chatgpt_df.rename(columns={
        'title': 'Publication_Title',
        'exposure': 'Exposure_ChatGPT',
        'outcome': 'Outcome_ChatGPT'
    })

    # Prepare results
    results = []
    
    for _, manual_row in table_s1_df.iterrows():
        pub_title = manual_row['Publication_Title']
        chatgpt_rows = chatgpt_df[chatgpt_df['Publication_Title'] == pub_title]
        
        if chatgpt_rows.empty:
            continue
            
        chatgpt_row = chatgpt_rows.iloc[0]
        
        # Check exposure against all layers
        exp_matches = {
            'Layer1': improved_matcher(chatgpt_row['Exposure_ChatGPT'], manual_row['Exposure_Layer1']),
            'Layer2': improved_matcher(chatgpt_row['Exposure_ChatGPT'], manual_row['Exposure_Layer2']),
            'Layer3': improved_matcher(chatgpt_row['Exposure_ChatGPT'], manual_row['Exposure_Layer3'])
        }
        
        # Check outcome against all layers
        out_matches = {
            'Layer1': improved_matcher(chatgpt_row['Outcome_ChatGPT'], manual_row['Outcome_Layer1']),
            'Layer2': improved_matcher(chatgpt_row['Outcome_ChatGPT'], manual_row['Outcome_Layer2']),
            'Layer3': improved_matcher(chatgpt_row['Outcome_ChatGPT'], manual_row['Outcome_Layer3'])
        }
        
        # Find best matches
        best_exp = max(exp_matches.items(), key=lambda x: x[1])
        best_out = max(out_matches.items(), key=lambda x: x[1])
        
        results.append({
            'Publication_Title': pub_title,
            'ChatGPT_Exposure': chatgpt_row['Exposure_ChatGPT'],
            'Manual_Exposure_Layer1': manual_row['Exposure_Layer1'],
            'Manual_Exposure_Layer2': manual_row['Exposure_Layer2'],
            'Manual_Exposure_Layer3': manual_row['Exposure_Layer3'],
            'Exposure_Match_Level': best_exp[0] if best_exp[1] else 'None',
            'Exposure_Match': best_exp[1],
            'ChatGPT_Outcome': chatgpt_row['Outcome_ChatGPT'],
            'Manual_Outcome_Layer1': manual_row['Outcome_Layer1'],
            'Manual_Outcome_Layer2': manual_row['Outcome_Layer2'],
            'Manual_Outcome_Layer3': manual_row['Outcome_Layer3'],
            'Outcome_Match_Level': best_out[0] if best_out[1] else 'None',
            'Outcome_Match': best_out[1],
            'Overall_Match': best_exp[1] and best_out[1]
        })

    # Create and save results
    results_df = pd.DataFrame(results)
    
    # Add fuzzy similarity scores
    results_df['Exposure_Similarity'] = results_df.apply(
        lambda x: fuzz.token_set_ratio(
            str(x['ChatGPT_Exposure']).lower(), 
            str(x[f'Manual_Exposure_{x["Exposure_Match_Level"]}']).lower()
        ) if x['Exposure_Match_Level'] != 'None' else 0, axis=1)
    
    results_df['Outcome_Similarity'] = results_df.apply(
        lambda x: fuzz.token_set_ratio(
            str(x['ChatGPT_Outcome']).lower(), 
            str(x[f'Manual_Outcome_{x["Outcome_Match_Level"]}']).lower()
        ) if x['Outcome_Match_Level'] != 'None' else 0, axis=1)
    
    # Save with encoding
    results_df.to_csv("improved_comparison_results.csv", index=False, encoding='utf-8')
    
    # Print diagnostics
    print("Improved comparison complete. Results saved to improved_comparison_results.csv")
    print("\nMatch Statistics:")
    print(f"- Exposure match rate: {results_df['Exposure_Match'].mean():.2%}")
    print(f"- Outcome match rate: {results_df['Outcome_Match'].mean():.2%}")
    print(f"- Overall match rate: {results_df['Overall_Match'].mean():.2%}")
    
    print("\nMatch Levels (when matched):")
    if results_df['Exposure_Match'].any():
        print("Exposure matches by level:")
        print(results_df[results_df['Exposure_Match']]['Exposure_Match_Level'].value_counts())
    if results_df['Outcome_Match'].any():
        print("\nOutcome matches by level:")
        print(results_df[results_df['Outcome_Match']]['Outcome_Match_Level'].value_counts())
    
    print("\nSimilarity Scores:")
    print(f"Average exposure similarity: {results_df['Exposure_Similarity'].mean():.1f}%")
    print(f"Average outcome similarity: {results_df['Outcome_Similarity'].mean():.1f}%")

except Exception as e:
    print(f"Error: {str(e)}")

Improved comparison complete. Results saved to improved_comparison_results.csv

Match Statistics:
- Exposure match rate: 69.15%
- Outcome match rate: 65.96%
- Overall match rate: 48.94%

Match Levels (when matched):
Exposure matches by level:
Exposure_Match_Level
Layer2    58
Layer3     6
Layer1     1
Name: count, dtype: int64

Outcome matches by level:
Outcome_Match_Level
Layer2    41
Layer3    12
Layer1     9
Name: count, dtype: int64

Similarity Scores:
Average exposure similarity: 66.4%
Average outcome similarity: 59.9%


/opt/anaconda3/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [31]:
import pandas as pd
import openai
import time


# Initialize OpenAI client
openai.api_key="YOUR_OPENAI_API_KEY_HERE" 

# Load the data
def load_data():
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print(f"Loaded data with {len(df)} records")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Define categorization prompt
def create_prompt(layer3_items, item_type="exposure"):
    return f"""
    Categorize these {item_type} terms from epidemiological studies into exactly 6 broad categories.
    The categories should be comprehensive enough to cover all terms but specific enough to be meaningful.
    
    For each term, provide:
    1. The original Layer 3 term
    2. Your assigned Layer 1 category
    3. A brief justification (1 sentence)
    
    Example:
    - Term: "short-term PM2.5 mass"
    - Category: "Air Pollution"
    - Justification: "PM2.5 is a well-known air pollutant"
    
    Here are the terms to categorize:
    {', '.join(layer3_items)}
    """

# Process data with ChatGPT
def categorize_with_chatgpt(df, item_type="exposure", col_name="Exposure_Layer3"):
    unique_terms = df[col_name].dropna().unique()
    print(f"Found {len(unique_terms)} unique {item_type} terms")
    
    results = []
    batch_size = 20  # Process terms in batches to avoid rate limits
    delay = 1  # seconds between requests
    
    for i in range(0, len(unique_terms), batch_size):
        batch = unique_terms[i:i+batch_size]
        prompt = create_prompt(batch, item_type)
        
        try:
            response = openai.api_key.chat.completions.create(
                model="gpt-4-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3
            )
            
            # Parse response
            content = response.choices[0].message.content
            results.append({
                'terms': batch,
                'response': content,
                'item_type': item_type
            })
            
            print(f"Processed batch {i//batch_size + 1}/{(len(unique_terms)//batch_size)+1}")
            time.sleep(delay)
            
        except Exception as e:
            print(f"Error processing batch {i//batch_size + 1}: {e}")
            continue
    
    return results

# Main execution
if __name__ == "__main__":
    df = load_data()
    if df is not None:
        # Process exposures
        exposure_results = categorize_with_chatgpt(df, "exposure", "Exposure_Layer3")
        exposure_df = pd.DataFrame(exposure_results)
        exposure_df.to_csv("exposure_categorizations.csv", index=False)
        
        # Process outcomes
        outcome_results = categorize_with_chatgpt(df, "outcome", "Outcome_Layer3")
        outcome_df = pd.DataFrame(outcome_results)
        outcome_df.to_csv("outcome_categorizations.csv", index=False)
        
        print("Categorization complete. Results saved to:")
        print("- exposure_categorizations.csv")
        print("- outcome_categorizations.csv")

Loaded data with 428 records
Found 114 unique exposure terms
Error processing batch 1: 'str' object has no attribute 'chat'
Error processing batch 2: 'str' object has no attribute 'chat'
Error processing batch 3: 'str' object has no attribute 'chat'
Error processing batch 4: 'str' object has no attribute 'chat'
Error processing batch 5: 'str' object has no attribute 'chat'
Error processing batch 6: 'str' object has no attribute 'chat'
Found 86 unique outcome terms
Error processing batch 1: 'str' object has no attribute 'chat'
Error processing batch 2: 'str' object has no attribute 'chat'
Error processing batch 3: 'str' object has no attribute 'chat'
Error processing batch 4: 'str' object has no attribute 'chat'
Error processing batch 5: 'str' object has no attribute 'chat'
Categorization complete. Results saved to:
- exposure_categorizations.csv
- outcome_categorizations.csv


In [33]:
import pandas as pd
import openai
import time
import os

# Configure OpenAI - using the older API style
openai.api_key = os.getenv("YOUR_OPENAI_API_KEY_HERE")  

def load_data():
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print(f"Loaded data with {len(df)} records")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

def create_prompt(layer3_items, item_type):
    return f"""
    Analyze these epidemiological {item_type} terms and categorize each into one of exactly 6 broad categories.
    The categories must be:
    1. Chemical Exposures
    2. Physical Exposures
    3. Biological Factors
    4. Behavioral Factors
    5. Psychosocial Factors
    6. Health Outcomes

    For each term, provide:
    - Original term
    - Assigned category
    - Brief justification (1 sentence)

    Return the results in CSV format with these columns:
    original_term,category,justification

    Example:
    "short-term PM2.5 mass","Chemical Exposures","PM2.5 is a chemical air pollutant"

    Terms to categorize:
    {', '.join(f'"{term}"' for term in layer3_items if pd.notna(term))}
    """

def process_terms(terms, item_type):
    results = []
    batch_size = 5  # Smaller batches for reliability
    delay = 3  # Longer delay between requests
    
    for i in range(0, len(terms), batch_size):
        batch = terms[i:i+batch_size]
        prompt = create_prompt(batch, item_type)
        
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",  # Using more accessible model
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            
            content = response.choices[0].message.content.strip()
            print(f"Received response for batch {i//batch_size + 1}:\n{content[:200]}...")  # Print first 200 chars
            
            # Parse CSV response
            lines = content.split('\n')
            for line in lines:
                if line.count(',') >= 2 and not line.startswith('original_term'):
                    parts = [p.strip('"').strip() for p in line.split(',', 2)]
                    if len(parts) == 3:
                        results.append({
                            'original_term': parts[0],
                            'category': parts[1],
                            'justification': parts[2]
                        })
            
            time.sleep(delay)
            
        except Exception as e:
            print(f"Error processing batch starting with '{batch[0]}': {str(e)}")
            time.sleep(5)  # Longer wait after errors
            continue
    
    return pd.DataFrame(results)

def main():
    df = load_data()
    if df is None:
        return
    
    # Test with a small sample first
    test_terms = ["short-term PM2.5 mass", "blood pressure", "optimism"]
    print("\nTesting with sample terms...")
    test_results = process_terms(test_terms, "test")
    print(test_results)
    
    if not test_results.empty:
        # Process all exposures
        exposure_terms = df['Exposure_Layer3'].dropna().unique()
        print(f"\nProcessing {len(exposure_terms)} exposure terms...")
        exposure_cats = process_terms(exposure_terms, "exposure")
        exposure_cats.to_csv("exposure_categories.csv", index=False)
        
        # Process all outcomes
        outcome_terms = df['Outcome_Layer3'].dropna().unique()
        print(f"\nProcessing {len(outcome_terms)} outcome terms...")
        outcome_cats = process_terms(outcome_terms, "outcome")
        outcome_cats.to_csv("outcome_categories.csv", index=False)
        
        print("\nResults saved to:")
        print(f"- exposure_categories.csv ({len(exposure_cats)} terms)")
        print(f"- outcome_categories.csv ({len(outcome_cats)} terms)")
    else:
        print("\nTest failed. Check API connectivity before processing full dataset.")

if __name__ == "__main__":
    main()

Loaded data with 428 records

Testing with sample terms...
Error processing batch starting with 'short-term PM2.5 mass': No API key provided. You can set your API key in code using 'openai.api_key = <API-KEY>', or you can set the environment variable OPENAI_API_KEY=<API-KEY>). If your API key is stored in a file, you can point the openai module at it with 'openai.api_key_path = <PATH>'. You can generate API keys in the OpenAI web interface. See https://platform.openai.com/account/api-keys for details.
Empty DataFrame
Columns: []
Index: []

Test failed. Check API connectivity before processing full dataset.


In [35]:
import openai
openai.api_key = "YOUR_OPENAI_API_KEY_HERE"

try:
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "Say 'test'"}]
    )
    print("API test successful:", response.choices[0].message.content)
except Exception as e:
    print("API test failed:", str(e))

API test successful: test


In [51]:
OPENAI_API_KEY="YOUR_OPENAI_API_KEY_HERE"

python script.py

SyntaxError: invalid syntax (2889534484.py, line 3)

In [47]:
import pandas as pd
import openai
import time
import os

# Remove this line - it's exposing your API key
# openai.api_key="YOUR_OPENAI_API_KEY_HERE" 

# Instead, set your API key safely:
openai.api_key = os.getenv("OPENAI_API_KEY")  # Set this in your environment first

def load_data():
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print(f"Loaded data with {len(df)} records")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

def create_prompt(layer3_items, item_type):
    return f"""
    Categorize these {item_type} terms into exactly 6 categories:
    1. Chemical Exposures
    2. Physical Exposures
    3. Biological Factors
    4. Behavioral Factors
    5. Psychosocial Factors
    6. Health Outcomes

    For each term provide:
    - Original term
    - Assigned category
    - Brief justification

    Return as CSV with columns: original_term,category,justification

    Example:
    "short-term PM2.5 mass","Chemical Exposures","PM2.5 is an air pollutant"

    Terms to categorize:
    {', '.join(f'"{term}"' for term in layer3_items if pd.notna(term))}
    """

def process_terms(terms, item_type):
    results = []
    batch_size = 5  # Small batches
    delay = 3  # Seconds between calls
    
    for i in range(0, len(terms), batch_size):
        batch = terms[i:i+batch_size]
        prompt = create_prompt(batch, item_type)
        
        try:
            # Correct API call syntax:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            
            content = response.choices[0].message.content
            print(f"Batch {i//batch_size + 1} response received")
            
            # Parse the CSV response
            for line in content.split('\n'):
                if line.count(',') >= 2 and not line.startswith('original_term'):
                    parts = [p.strip().strip('"') for p in line.split(',', 2)]
                    if len(parts) == 3:
                        results.append({
                            'original_term': parts[0],
                            'category': parts[1],
                            'justification': parts[2]
                        })
            
            time.sleep(delay)
            
        except Exception as e:
            print(f"Error on batch {i//batch_size + 1}: {str(e)}")
            time.sleep(5)
            continue
    
    return pd.DataFrame(results)

def main():
    df = load_data()
    if df is None:
        return
    
    # First verify API connection with test terms
    test_terms = ["short-term PM2.5 mass", "blood pressure", "optimism"]
    print("\nTesting API connection...")
    test_df = process_terms(test_terms, "test")
    
    if test_df.empty:
        print("\nAPI test failed. Please check:")
        print("1. Your OpenAI API key is set in environment variables")
        print("2. You have sufficient API credits")
        print("3. Your network connection is working")
        return
    
    print("\nAPI test successful. Processing full dataset...")
    
    # Process exposures
    exposure_terms = df['Exposure_Layer3'].dropna().unique()
    print(f"\nProcessing {len(exposure_terms)} exposure terms...")
    exposure_df = process_terms(exposure_terms, "exposure")
    exposure_df.to_csv("exposure_categories.csv", index=False)
    
    # Process outcomes
    outcome_terms = df['Outcome_Layer3'].dropna().unique()
    print(f"\nProcessing {len(outcome_terms)} outcome terms...")
    outcome_df = process_terms(outcome_terms, "outcome")
    outcome_df.to_csv("outcome_categories.csv", index=False)
    
    print("\nProcessing complete. Results saved to:")
    print(f"- exposure_categories.csv ({len(exposure_df)} terms)")
    print(f"- outcome_categories.csv ({len(outcome_df)} terms)")

if __name__ == "__main__":
    main()

Loaded data with 428 records

Testing API connection...
Error on batch 1: No API key provided. You can set your API key in code using 'openai.api_key = <API-KEY>', or you can set the environment variable OPENAI_API_KEY=<API-KEY>). If your API key is stored in a file, you can point the openai module at it with 'openai.api_key_path = <PATH>'. You can generate API keys in the OpenAI web interface. See https://platform.openai.com/account/api-keys for details.

API test failed. Please check:
1. Your OpenAI API key is set in environment variables
2. You have sufficient API credits
3. Your network connection is working


In [53]:
import pandas as pd
import openai
import time
import os
from dotenv import load_dotenv

# 1. First try loading from .env file
load_dotenv()

# 2. set API key 
API_KEY = (
    os.getenv("OPENAI_API_KEY") or       
    "YOUR_OPENAI_API_KEY_HERE"                     
)

if not API_KEY:
    raise ValueError("No OpenAI API key found. Please set OPENAI_API_KEY environment variable")

openai.api_key = API_KEY

def load_data():
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print(f"Loaded data with {len(df)} records")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

def create_prompt(layer3_items, item_type):
    return f"""
    Categorize these {item_type} terms into exactly 6 categories:
    1. Chemical Exposures
    2. Physical Exposures
    3. Biological Factors
    4. Behavioral Factors
    5. Psychosocial Factors
    6. Health Outcomes

    Return as CSV with columns: original_term,category,justification

    Example:
    "short-term PM2.5 mass","Chemical Exposures","PM2.5 is an air pollutant"

    Terms to categorize:
    {', '.join(f'"{term}"' for term in layer3_items if pd.notna(term))}
    """

def process_terms(terms, item_type):
    results = []
    batch_size = 3  # Very small batches for reliability
    delay = 5  # Conservative delay
    
    for i in range(0, len(terms), batch_size):
        batch = terms[i:i+batch_size]
        prompt = create_prompt(batch, item_type)
        
        try:
            print(f"\nProcessing batch {i//batch_size + 1}: {batch}")
            
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            
            content = response.choices[0].message.content
            print("API Response Received")
            
            # Parse response
            for line in content.split('\n'):
                if line.count(',') >= 2 and line.strip() and not line.startswith('original_term'):
                    parts = [p.strip().strip('"') for p in line.split(',', 2)]
                    if len(parts) == 3:
                        results.append({
                            'original_term': parts[0],
                            'category': parts[1],
                            'justification': parts[2]
                        })
                        print(f"Processed: {parts[0]} → {parts[1]}")
            
            time.sleep(delay)
            
        except Exception as e:
            print(f"Error processing batch: {str(e)}")
            print("Waiting 10 seconds before retry...")
            time.sleep(10)
            continue
    
    return pd.DataFrame(results)

def main():
    # Verify API connection first
    print("Testing API connection...")
    try:
        test_response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": "Say 'TEST'"}],
            max_tokens=5
        )
        print("API Connection Successful:", test_response.choices[0].message.content)
    except Exception as e:
        print(f"API Connection Failed: {str(e)}")
        print("\nTroubleshooting:")
        print(f"1. Current API key: {'Set' if API_KEY else 'NOT SET'}")
        print("2. Try setting key directly in code (temporarily)")
        print("3. Check https://platform.openai.com/account/usage for credits")
        return
    
    # Load and process data
    df = load_data()
    if df is None:
        return
    
    # Process exposures
    exposure_terms = df['Exposure_Layer3'].dropna().unique()[:10]  # First 10 for testing
    print(f"\nProcessing {len(exposure_terms)} exposure terms...")
    exposure_df = process_terms(exposure_terms, "exposure")
    
    if not exposure_df.empty:
        exposure_df.to_csv("exposure_categories.csv", index=False)
        print(f"\nSaved {len(exposure_df)} exposure categorizations")
    else:
        print("\nNo exposure categorizations generated")

if __name__ == "__main__":
    main()

Testing API connection...
API Connection Successful: TEST
Loaded data with 428 records

Processing 10 exposure terms...

Processing batch 1: ['short-term PM2.5 mass' 'short-term PM2.5 elemental components'
 'short-term black carbon']
API Response Received
Processed: short-term PM2.5 mass → Chemical Exposures
Processed: short-term PM2.5 elemental components → Chemical Exposures
Processed: short-term black carbon → Chemical Exposures

Processing batch 2: ['intermediate-term PM2.5 mass'
 'intermediate-term PM2.5 elemental components'
 'intermediate-term black carbon']
API Response Received
Processed: intermediate-term PM2.5 mass → Chemical Exposures
Processed: intermediate-term PM2.5 elemental components → Chemical Exposures
Processed: intermediate-term black carbon → Chemical Exposures

Processing batch 3: ['interplanetary magnetic field' 'sunspot number'
 'geomagnetic activity-K index']
API Response Received
Processed: interplanetary magnetic field → Physical Exposures
Processed: sunspo

In [55]:
import pandas as pd
import openai
import time
import os
from tqdm import tqdm
from dotenv import load_dotenv

# Configure API
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE"

def load_data():
    """Load and preprocess the dataset"""
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        print(f"✅ Loaded {len(df)} records")
        return df
    except Exception as e:
        print(f"❌ Data loading failed: {e}")
        return None

def create_prompt(terms, item_type):
    """Generate optimized prompt for categorization"""
    return f"""
    Categorize these {item_type} terms into exactly 6 categories:
    1. Chemical Exposures | 2. Physical Exposures | 3. Biological Factors
    4. Behavioral Factors | 5. Psychosocial Factors | 6. Health Outcomes

    Return CSV format: original_term,category,justification

    Examples:
    "PM2.5","Chemical Exposures","Particulate matter air pollution"
    "blood pressure","Health Outcomes","Cardiovascular health metric"

    Terms:
    {', '.join(f'"{t}"' for t in terms if pd.notna(t))}
    """

def process_batch(terms, item_type, retries=3):
    """Process a batch with automatic retries"""
    for attempt in range(retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{
                    "role": "user", 
                    "content": create_prompt(terms, item_type)
                }],
                temperature=0.1
            )
            return parse_response(response.choices[0].message.content)
        except Exception as e:
            if attempt == retries - 1:
                print(f"❌ Failed batch after {retries} attempts: {e}")
                return []
            wait = (attempt + 1) * 5
            print(f"⚠️ Retry {attempt + 1} in {wait}s...")
            time.sleep(wait)

def parse_response(content):
    """Parse API response into structured data"""
    results = []
    for line in content.split('\n'):
        if line.count(',') >= 2 and not line.startswith('original_term'):
            parts = [p.strip().strip('"') for p in line.split(',', 2)]
            if len(parts) == 3:
                results.append({
                    'original_term': parts[0],
                    'category': parts[1], 
                    'justification': parts[2]
                })
    return results

def process_all_terms(terms, item_type):
    """Process all terms with progress tracking"""
    batch_size = 5  # Optimal for GPT-3.5
    results = []
    
    with tqdm(total=len(terms), desc=f"Processing {item_type}s") as pbar:
        for i in range(0, len(terms), batch_size):
            batch = terms[i:i + batch_size]
            batch_results = process_batch(batch, item_type)
            results.extend(batch_results)
            pbar.update(len(batch))
            time.sleep(2)  # Respect rate limits
    
    return pd.DataFrame(results)

def main():
    # Load data
    df = load_data()
    if df is None:
        return

    # Process exposures
    exposure_terms = df['Exposure_Layer3'].dropna().unique()
    exposure_df = process_all_terms(exposure_terms, "exposure")
    exposure_df.to_csv("exposure_categories_full.csv", index=False)
    
    # Process outcomes
    outcome_terms = df['Outcome_Layer3'].dropna().unique()
    outcome_df = process_all_terms(outcome_terms, "outcome") 
    outcome_df.to_csv("outcome_categories_full.csv", index=False)
    
    print("\n🎉 Processing complete!")
    print(f"Exposure categories: {len(exposure_df)} terms")
    print(f"Outcome categories: {len(outcome_df)} terms")

if __name__ == "__main__":
    main()

✅ Loaded 428 records


Processing outcomes: 100%|██████████████████████| 86/86 [00:56<00:00,  1.52it/s]


🎉 Processing complete!
Exposure categories: 114 terms
Outcome categories: 86 terms


In [61]:
import pandas as pd
import openai
import time
import os
from tqdm import tqdm
from dotenv import load_dotenv
import requests
import json

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE"
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 3  # seconds

def load_reference_data():
    """Load and clean the reference data"""
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        
        # Clean titles and handle NaN values
        df['clean_title'] = df['Publication_Title'].str.lower().str.replace('[^a-z0-9 ]', '').fillna('')
        print(f"✅ Loaded reference data: {len(df)} records")
        return df
    except Exception as e:
        print(f"❌ Failed to load reference data: {e}")
        return None

def search_pubmed(title, retries=3):
    """Robust PubMed search with retries"""
    if pd.isna(title) or not title.strip():
        return None
        
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        'db': 'pubmed',
        'term': f'"{title}"[Title]',
        'retmode': 'json',
        'retmax': 1
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            if int(data.get('esearchresult', {}).get('count', 0)) > 0:
                return data['esearchresult']['idlist'][0]
            return None
        except Exception as e:
            if attempt == retries - 1:
                print(f"⚠️ PubMed search failed for '{title[:30]}...': {e}")
            time.sleep(2)
    return None

def fetch_article(pmid, retries=3):
    """Fetch article with retry logic"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        'db': 'pubmed',
        'id': pmid,
        'retmode': 'text',
        'rettype': 'abstract'
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            return response.text
        except Exception as e:
            if attempt == retries - 1:
                print(f"⚠️ Failed to fetch PMID {pmid}: {e}")
            time.sleep(2)
    return None

def analyze_with_chatgpt(content, title, max_tokens=1000):
    """Analyze content with robust error handling"""
    if not content or not title:
        return None
        
    prompt = f"""
    Analyze this research paper and extract key elements:
    
    EXPOSURES (what was studied as potential causes):
    - List specific exposures (e.g., "PM2.5", "lead levels")
    - Maximum 3-5 most important exposures
    
    OUTCOMES (what health effects were measured):
    - List specific outcomes (e.g., "lung function", "cognitive scores")
    - Maximum 3-5 most important outcomes
    
    Return JSON format:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}
    
    Paper Title: {title[:200]}
    Content: {content[:8000]} [truncated]
    """
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",  # More cost-effective
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=max_tokens,
                request_timeout=30
            )
            return json.loads(response.choices[0].message.content)
        except openai.error.RateLimitError:
            print(f"⏳ Rate limited, waiting {DELAY_BETWEEN_REQUESTS*2} seconds...")
            time.sleep(DELAY_BETWEEN_REQUESTS * 2)
        except Exception as e:
            print(f"⚠️ Analysis attempt {attempt+1} failed for '{title[:30]}...': {str(e)[:100]}")
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(DELAY_BETWEEN_REQUESTS)
    return None

def process_articles(reference_df, limit=None):
    """Main processing pipeline with progress tracking"""
    results = []
    processed_count = 0
    
    for _, row in tqdm(reference_df.iterrows(), total=len(reference_df), desc="Processing papers"):
        title = str(row['Publication_Title']) if pd.notna(row['Publication_Title']) else ""
        
        # Skip if title is empty
        if not title.strip():
            continue
            
        # Find PubMed ID
        pmid = search_pubmed(title)
        if not pmid:
            continue
            
        # Fetch content
        content = fetch_article(pmid)
        if not content:
            continue
            
        # Analyze with ChatGPT
        analysis = analyze_with_chatgpt(content, title)
        if not analysis:
            continue
            
        # Prepare results
        result = {
            'original_title': title,
            'pmid': pmid,
            'extracted_exposures': analysis.get('exposures', []),
            'extracted_outcomes': analysis.get('outcomes', []),
            'reference_exposure': row['Exposure_Layer3'] if pd.notna(row['Exposure_Layer3']) else "",
            'reference_outcome': row['Outcome_Layer3'] if pd.notna(row['Outcome_Layer3']) else "",
            'exposure_match': any(
                isinstance(row['Exposure_Layer3'], str) and 
                any(exposure.lower() in row['Exposure_Layer3'].lower() 
                    for exposure in analysis.get('exposures', []))
            ),
            'outcome_match': any(
                isinstance(row['Outcome_Layer3'], str) and 
                any(outcome.lower() in row['Outcome_Layer3'].lower() 
                    for outcome in analysis.get('outcomes', []))
            )
        }
        results.append(result)
        processed_count += 1
        
        # Optional early stopping
        if limit and processed_count >= limit:
            break
            
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return pd.DataFrame(results)

def main():
    # Load reference data
    reference_df = load_reference_data()
    if reference_df is None:
        return
    
    # Process articles (with optional limit for testing)
    results_df = process_articles(reference_df, limit=10)  # Remove limit for full processing
    
    # Save results
    if not results_df.empty:
        results_df.to_csv("extracted_vs_reference.csv", index=False)
        print(f"\n🎉 Completed! Results saved for {len(results_df)} papers")
        
        # Calculate match rates
        exposure_match = results_df['exposure_match'].mean()
        outcome_match = results_df['outcome_match'].mean()
        print(f"\nMatch Rates:")
        print(f"- Exposures: {exposure_match:.1%}")
        print(f"- Outcomes: {outcome_match:.1%}")
    else:
        print("\n❌ No results generated - check errors above")

if __name__ == "__main__":
    main()

✅ Loaded reference data: 428 records


Processing papers: 100%|██████████████████████| 428/428 [01:45<00:00,  4.06it/s]


❌ No results generated - check errors above


In [63]:
import pandas as pd
import openai
import time
import os
from tqdm import tqdm
from dotenv import load_dotenv
import requests
import json

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE"
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 2  # seconds

def load_reference_data():
    """Load and clean the reference data"""
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        
        # Clean titles and handle duplicates
        df['clean_title'] = df['Publication_Title'].str.lower().str.replace('[^a-z0-9 ]', '').str.strip().fillna('')
        
        # Get unique papers based on cleaned titles
        unique_papers = df.drop_duplicates('clean_title', keep='first')
        print(f"✅ Loaded {len(df)} records, found {len(unique_papers)} unique papers")
        return unique_papers
    except Exception as e:
        print(f"❌ Failed to load reference data: {e}")
        return None

def search_pubmed(title, retries=3):
    """Search PubMed for exact title match"""
    if not title or not isinstance(title, str):
        return None
        
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        'db': 'pubmed',
        'term': f'"{title}"[Title]',
        'retmode': 'json',
        'retmax': 1
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            if int(data.get('esearchresult', {}).get('count', 0)) > 0:
                return data['esearchresult']['idlist'][0]
            return None
        except Exception as e:
            if attempt == retries - 1:
                print(f"⚠️ PubMed search failed for '{title[:30]}...': {e}")
            time.sleep(1)
    return None

def fetch_article(pmid, retries=3):
    """Fetch article abstract with retry logic"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        'db': 'pubmed',
        'id': pmid,
        'retmode': 'text',
        'rettype': 'abstract'
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            return response.text
        except Exception as e:
            if attempt == retries - 1:
                print(f"⚠️ Failed to fetch PMID {pmid}: {e}")
            time.sleep(1)
    return None

def analyze_with_chatgpt(content, title, max_tokens=1000):
    """Analyze paper content with robust error handling"""
    if not content or not title:
        return None
        
    prompt = f"""
    Extract from this research paper:
    
    1. EXPOSURES (what was studied as potential causes):
    - List 3-5 specific exposure terms
    - Example: "PM2.5 levels", "lead exposure"
    
    2. OUTCOMES (measured health effects):
    - List 3-5 specific outcome terms  
    - Example: "lung function", "cognitive scores"
    
    Return JSON format:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}
    
    Paper Title: {title[:200]}
    Content: {content[:8000]} [truncated]
    """
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=max_tokens,
                request_timeout=30
            )
            return json.loads(response.choices[0].message.content)
        except openai.error.RateLimitError:
            wait = DELAY_BETWEEN_REQUESTS * (attempt + 1)
            print(f"⏳ Rate limited, waiting {wait} seconds...")
            time.sleep(wait)
        except Exception as e:
            print(f"⚠️ Analysis attempt {attempt+1} failed: {str(e)[:100]}")
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(DELAY_BETWEEN_REQUESTS)
    return None

def process_unique_papers(unique_papers_df, limit=None):
    """Process unique papers with progress tracking"""
    results = []
    processed_count = 0
    
    for _, paper in tqdm(unique_papers_df.iterrows(), total=len(unique_papers_df), desc="Processing unique papers"):
        title = str(paper['Publication_Title']) if pd.notna(paper['Publication_Title']) else ""
        
        if not title.strip():
            continue
            
        # Find PubMed ID
        pmid = search_pubmed(title)
        if not pmid:
            print(f"⏩ No PubMed match for '{title[:50]}...'")
            continue
            
        # Fetch content
        content = fetch_article(pmid)
        if not content:
            continue
            
        # Analyze with ChatGPT
        analysis = analyze_with_chatgpt(content, title)
        if not analysis:
            continue
            
        # Prepare results
        result = {
            'original_title': title,
            'pmid': pmid,
            'extracted_exposures': analysis.get('exposures', []),
            'extracted_outcomes': analysis.get('outcomes', []),
            'reference_exposure': paper['Exposure_Layer3'] if pd.notna(paper['Exposure_Layer3']) else "",
            'reference_outcome': paper['Outcome_Layer3'] if pd.notna(paper['Outcome_Layer3']) else "",
            'exposure_match': any(
                isinstance(paper['Exposure_Layer3'], str) and 
                any(exposure.lower() in paper['Exposure_Layer3'].lower() 
                    for exposure in analysis.get('exposures', []))
            ),
            'outcome_match': any(
                isinstance(paper['Outcome_Layer3'], str) and 
                any(outcome.lower() in paper['Outcome_Layer3'].lower() 
                    for outcome in analysis.get('outcomes', []))
            )
        }
        results.append(result)
        processed_count += 1
        
        # Optional early stopping
        if limit and processed_count >= limit:
            break
            
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return pd.DataFrame(results)

def main():
    # Load reference data
    reference_df = load_reference_data()
    if reference_df is None:
        return
    
    # Process unique papers (with optional limit for testing)
    results_df = process_unique_papers(reference_df, limit=5)  # Remove limit for full processing
    
    # Save results
    if not results_df.empty:
        results_df.to_csv("paper_extraction_results.csv", index=False)
        print(f"\n🎉 Success! Processed {len(results_df)} unique papers")
        
        # Calculate match rates
        if len(results_df) > 0:
            exposure_match = results_df['exposure_match'].mean()
            outcome_match = results_df['outcome_match'].mean()
            print(f"\nMatch Rates:")
            print(f"- Exposures: {exposure_match:.1%}")
            print(f"- Outcomes: {outcome_match:.1%}")
            
            # Sample output
            print("\nSample extracted data:")
            print(results_df[['original_title', 'extracted_exposures', 'extracted_outcomes']].head(2))
    else:
        print("\n❌ No results generated - check errors above")

if __name__ == "__main__":
    main()

✅ Loaded 428 records, found 122 unique papers


Processing unique papers:   1%|▏                | 1/122 [00:00<00:28,  4.26it/s]

⏩ No PubMed match for 'short- and intermediate-term exposure to ambient f...'


Processing unique papers:   2%|▍                | 3/122 [00:00<00:17,  6.94it/s]

⏩ No PubMed match for 'associations between solar and geomagnetic activit...'


...

⏩ No PubMed match for 'repetitive element hypomethylation in blood leukoc...'


Processing unique papers:  99%|██████████████▉| 121/122 [01:15<00:00,  1.30it/s]

⏩ No PubMed match for 'traffic-related air pollution and cognitive functi...'


Processing unique papers: 100%|███████████████| 122/122 [01:16<00:00,  1.60it/s]

⏩ No PubMed match for 'urinary 8-hydroxy-2'-deoxyguanosine as a biomarker...'

❌ No results generated - check errors above


In [65]:
import pandas as pd
import openai
import time
import os
import re
import requests
import json
from tqdm import tqdm
from dotenv import load_dotenv
from fuzzywuzzy import fuzz
from urllib.parse import quote
import logging

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE"
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 2  # seconds
LOG_FILE = "pubmed_processing.log"

# Set up logging
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def clean_title(title):
    """Clean paper title while preserving essential punctuation"""
    if pd.isna(title):
        return ""
    title = str(title).lower()
    # Keep basic punctuation that might be in titles
    title = re.sub(r'[^\w\s\-&\',:;]', '', title)
    return title.strip()

def load_reference_data(csv_path):
    """Load and clean the reference data"""
    try:
        df = pd.read_csv(csv_path, encoding='latin1')
        
        # Clean titles while preserving important punctuation
        df['clean_title'] = df['Publication_Title'].apply(clean_title)
        
        # Get unique papers based on cleaned titles
        unique_papers = df.drop_duplicates('clean_title', keep='first')
        logging.info(f"Loaded {len(df)} records, found {len(unique_papers)} unique papers")
        print(f"✅ Loaded {len(df)} records, found {len(unique_papers)} unique papers")
        return unique_papers
    except Exception as e:
        logging.error(f"Failed to load reference data: {e}")
        print(f"❌ Failed to load reference data: {e}")
        return None

def search_pubmed(title, retries=3):
    """Search PubMed with exact match and fuzzy matching fallback"""
    if not title or not isinstance(title, str):
        return None
        
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    
    # First try exact match
    exact_pmid = try_exact_search(base_url, title)
    if exact_pmid:
        return exact_pmid
        
    # Fallback to fuzzy matching if exact fails
    return try_fuzzy_search(base_url, title)

def try_exact_search(base_url, title):
    """Try exact title match in PubMed"""
    params = {
        'db': 'pubmed',
        'term': quote(f'"{title}"[Title]'),
        'field': 'title',
        'retmode': 'json',
        'retmax': 1
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        if int(data.get('esearchresult', {}).get('count', 0)) > 0:
            return data['esearchresult']['idlist'][0]
    except Exception as e:
        logging.warning(f"Exact search failed for '{title[:50]}...': {str(e)[:100]}")
    return None

def try_fuzzy_search(base_url, title):
    """Fallback to fuzzy matching when exact match fails"""
    params = {
        'db': 'pubmed',
        'term': quote(f'{title}[Title]'),
        'retmode': 'json',
        'retmax': 5,
        'sort': 'relevance'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=20)
        response.raise_for_status()
        data = response.json()
        idlist = data.get('esearchresult', {}).get('idlist', [])
        
        # Fetch titles of top matches to compare
        for pmid in idlist[:3]:  # Check top 3 matches
            content = fetch_article(pmid)
            if content:
                # Extract title from abstract text
                match = re.search(r'TI\s*-\s*(.+?)\n', content)
                if match:
                    pubmed_title = match.group(1).lower()
                    similarity = fuzz.ratio(title.lower(), pubmed_title)
                    if similarity > 85:  # Good enough match
                        logging.info(f"Fuzzy match found ({similarity}%): {title[:50]}...")
                        return pmid
    except Exception as e:
        logging.warning(f"Fuzzy search failed for '{title[:50]}...': {str(e)[:100]}")
    return None

def fetch_article(pmid, retries=3):
    """Fetch full article text from PubMed"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        'db': 'pubmed',
        'id': pmid,
        'retmode': 'text',
        'rettype': 'abstract'
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=15)
            response.raise_for_status()
            return response.text
        except Exception as e:
            if attempt == retries - 1:
                logging.error(f"Failed to fetch PMID {pmid}: {e}")
            time.sleep(1)
    return None

def extract_key_elements(content, title):
    """Use ChatGPT to extract exposures and outcomes from paper content"""
    if not content or not title:
        return None
        
    prompt = f"""
    Analyze this scientific paper and extract:
    
    1. EXPOSURES (independent variables/interventions studied):
    - List 3-5 specific exposure terms
    - Include exact chemical names, pollutants, treatments
    - Example: "PM2.5 levels", "lead exposure (Pb)"
    
    2. OUTCOMES (dependent variables/measured effects):
    - List 3-5 specific outcome terms  
    - Include exact health measures, biomarkers
    - Example: "FEV1 lung function", "MMSE cognitive scores"
    
    Return JSON format only:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}
    
    Paper Title: {title[:200]}
    Content: {content[:12000]} [truncated]
    """
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo-1106",  # Updated model
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,  # More deterministic output
                max_tokens=500,
                response_format={ "type": "json_object" },  # Ensure JSON output
                request_timeout=45
            )
            return json.loads(response.choices[0].message.content)
        except openai.error.RateLimitError:
            wait = DELAY_BETWEEN_REQUESTS * (attempt + 1)
            logging.warning(f"Rate limited, waiting {wait} seconds...")
            time.sleep(wait)
        except Exception as e:
            logging.error(f"Analysis attempt {attempt+1} failed: {str(e)[:100]}")
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(DELAY_BETWEEN_REQUESTS)
    return None

def compare_with_reference(extracted, reference):
    """Compare extracted elements with reference data from CSV"""
    matches = {
        'exposure_match': False,
        'outcome_match': False,
        'exposure_matches': [],
        'outcome_matches': []
    }
    
    # Compare exposures
    if isinstance(reference.get('Exposure_Layer3'), str):
        ref_exposures = [x.strip().lower() for x in reference['Exposure_Layer3'].split(';')]
        for exp in extracted.get('exposures', []):
            exp_lower = exp.lower()
            for ref_exp in ref_exposures:
                if ref_exp in exp_lower or exp_lower in ref_exp:
                    matches['exposure_match'] = True
                    matches['exposure_matches'].append((exp, ref_exp))
    
    # Compare outcomes
    if isinstance(reference.get('Outcome_Layer3'), str):
        ref_outcomes = [x.strip().lower() for x in reference['Outcome_Layer3'].split(';')]
        for out in extracted.get('outcomes', []):
            out_lower = out.lower()
            for ref_out in ref_outcomes:
                if ref_out in out_lower or out_lower in ref_out:
                    matches['outcome_match'] = True
                    matches['outcome_matches'].append((out, ref_out))
    
    return matches

def process_papers(unique_papers_df, limit=None):
    """Process papers with enhanced error handling"""
    results = []
    processed_count = 0
    
    for _, paper in tqdm(unique_papers_df.iterrows(), total=len(unique_papers_df), desc="Processing papers"):
        title = str(paper['Publication_Title']) if pd.notna(paper['Publication_Title']) else ""
        
        if not title.strip():
            continue
            
        # Find PubMed ID (with fuzzy matching fallback)
        pmid = search_pubmed(title)
        if not pmid:
            logging.info(f"No PubMed match for '{title[:50]}...'")
            print(f"⏩ No PubMed match for '{title[:50]}...'")
            continue
            
        # Fetch full article content
        content = fetch_article(pmid)
        if not content:
            continue
            
        # Extract key elements using ChatGPT
        analysis = extract_key_elements(content, title)
        if not analysis:
            continue
            
        # Compare with reference data from CSV
        comparison = compare_with_reference(analysis, paper)
        
        # Prepare results
        result = {
            'original_title': title,
            'pmid': pmid,
            'extracted_exposures': analysis.get('exposures', []),
            'extracted_outcomes': analysis.get('outcomes', []),
            'reference_exposure': paper.get('Exposure_Layer3', ''),
            'reference_outcome': paper.get('Outcome_Layer3', ''),
            'exposure_match': comparison['exposure_match'],
            'outcome_match': comparison['outcome_match'],
            'exposure_matches': '; '.join([f"{x[0]}≈{x[1]}" for x in comparison['exposure_matches']]),
            'outcome_matches': '; '.join([f"{x[0]}≈{x[1]}" for x in comparison['outcome_matches']]),
            'content_length': len(content)
        }
        results.append(result)
        processed_count += 1
        
        # Save progress periodically
        if processed_count % 5 == 0:
            pd.DataFrame(results).to_csv("progress_backup.csv", index=False)
        
        # Optional early stopping
        if limit and processed_count >= limit:
            break
            
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return pd.DataFrame(results)

def generate_report(results_df):
    """Generate summary report with statistics"""
    if results_df.empty:
        return "No results to report"
    
    report = []
    
    # Basic stats
    total = len(results_df)
    exposure_match_rate = results_df['exposure_match'].mean()
    outcome_match_rate = results_df['outcome_match'].mean()
    
    report.append(f"📊 Processing Report")
    report.append(f"Total papers processed: {total}")
    report.append(f"Exposure match rate: {exposure_match_rate:.1%}")
    report.append(f"Outcome match rate: {outcome_match_rate:.1%}")
    
    # Example matches
    report.append("\n🔍 Sample matches:")
    for _, row in results_df.head(3).iterrows():
        report.append(f"\nTitle: {row['original_title'][:60]}...")
        report.append(f"Exposures: {row['extracted_exposures']}")
        report.append(f"Outcomes: {row['extracted_outcomes']}")
        report.append(f"Matches: {row['exposure_matches'] or 'None'} (exp), {row['outcome_matches'] or 'None'} (out)")
    
    # Common mismatches
    mismatches = results_df[~results_df['exposure_match'] | ~results_df['outcome_match']]
    if not mismatches.empty:
        report.append("\n⚠️ Common mismatch patterns:")
        for _, row in mismatches.head(3).iterrows():
            report.append(f"\nTitle: {row['original_title'][:60]}...")
            report.append(f"Reference: {row['reference_exposure']} → {row['reference_outcome']}")
            report.append(f"Extracted: {row['extracted_exposures']} → {row['extracted_outcomes']}")
    
    return "\n".join(report)

def main():
    csv_path = "CohortNetwork_ES&T_SI_B_Main(Table S1).csv"
    
    # Load reference data
    reference_df = load_reference_data(csv_path)
    if reference_df is None:
        return
    
    # Process papers (remove limit=None for full processing)
    results_df = process_papers(reference_df, limit=None)
    
    # Save results
    if not results_df.empty:
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
        output_file = f"extraction_results_{timestamp}.csv"
        results_df.to_csv(output_file, index=False)
        
        # Generate and save report
        report = generate_report(results_df)
        with open(f"report_{timestamp}.txt", 'w') as f:
            f.write(report)
        
        print(f"\n🎉 Success! Results saved to {output_file}")
        print("\n" + report)
    else:
        print("\n❌ No results generated - check log file for details")

if __name__ == "__main__":
    main()

✅ Loaded 428 records, found 122 unique papers


Processing papers:   1%|▏                       | 1/122 [00:00<01:49,  1.10it/s]

⏩ No PubMed match for 'short- and intermediate-term exposure to ambient f...'


Processing papers:   2%|▌                       | 3/122 [00:01<00:57,  2.08it/s]

⏩ No PubMed match for 'associations between solar and geomagnetic activit...'


...

⏩ No PubMed match for 'repetitive element hypomethylation in blood leukoc...'


Processing papers:  99%|█████████████████████▊| 121/122 [01:22<00:00,  1.32it/s]

⏩ No PubMed match for 'traffic-related air pollution and cognitive functi...'


Processing papers: 100%|██████████████████████| 122/122 [01:23<00:00,  1.46it/s]

⏩ No PubMed match for 'urinary 8-hydroxy-2'-deoxyguanosine as a biomarker...'

❌ No results generated - check log file for details


In [67]:
import re
from fuzzywuzzy import fuzz
from urllib.parse import quote
import logging

def clean_title(title):
    """Clean paper title while preserving essential punctuation"""
    if pd.isna(title):
        return ""
    title = str(title).lower()
    # Keep basic punctuation that might be in titles
    title = re.sub(r'[^\w\s\-&\',:;\(\)]', '', title)  # Added parentheses
    return title.strip()

def search_pubmed(title, retries=3, similarity_threshold=80):
    """Enhanced PubMed search with flexible matching"""
    if not title or not isinstance(title, str):
        return None
        
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    clean_title = prepare_title_for_search(title)
    
    # Strategy 1: Exact match with original title
    exact_pmid = try_search_strategy(base_url, f'"{clean_title}"[Title]')
    if exact_pmid:
        return exact_pmid
        
    # Strategy 2: Match without quotes (partial match)
    partial_pmid = try_search_strategy(base_url, f'{clean_title}[Title]')
    if partial_pmid:
        return partial_pmid
        
    # Strategy 3: Split long titles and search first part
    if len(clean_title.split()) > 8:
        first_part = ' '.join(clean_title.split()[:8])
        partial_pmid = try_search_strategy(base_url, f'"{first_part}"[Title]')
        if partial_pmid:
            return partial_pmid
    
    # Strategy 4: Fuzzy match with top results
    return try_fuzzy_search(base_url, clean_title, similarity_threshold)

def prepare_title_for_search(title):
    """Prepare title for PubMed search"""
    # Remove common problematic patterns
    title = re.sub(r'\s+', ' ', title)  # Collapse multiple spaces
    title = re.sub(r'\.\.\.', '', title)  # Remove ellipses
    # Keep only first 150 characters to avoid API limits
    return title[:150].strip()

def try_search_strategy(base_url, search_term):
    """Try a search strategy with the given term"""
    params = {
        'db': 'pubmed',
        'term': quote(search_term),
        'retmode': 'json',
        'retmax': 1,
        'sort': 'relevance',
        'field': 'title'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        if int(data.get('esearchresult', {}).get('count', 0)) > 0:
            return data['esearchresult']['idlist'][0]
    except Exception as e:
        logging.debug(f"Search failed for '{search_term[:50]}...': {str(e)[:100]}")
    return None

def try_fuzzy_search(base_url, original_title, similarity_threshold):
    """Fallback to fuzzy matching with similarity threshold"""
    # First get candidate articles
    params = {
        'db': 'pubmed',
        'term': quote(f'{original_title}[Title]'),
        'retmode': 'json',
        'retmax': 5,
        'sort': 'relevance'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=20)
        response.raise_for_status()
        data = response.json()
        idlist = data.get('esearchresult', {}).get('idlist', [])
        
        # Check each candidate
        for pmid in idlist:
            content = fetch_article(pmid)
            if not content:
                continue
                
            # Extract title from content
            pubmed_title = extract_title_from_content(content)
            if not pubmed_title:
                continue
                
            # Calculate similarity
            similarity = fuzz.ratio(original_title.lower(), pubmed_title.lower())
            if similarity >= similarity_threshold:
                logging.info(f"Fuzzy match ({similarity}%): '{original_title[:50]}...' = '{pubmed_title[:50]}...'")
                return pmid
                
    except Exception as e:
        logging.warning(f"Fuzzy search failed: {str(e)[:100]}")
    
    return None

def extract_title_from_content(content):
    """Extract title from PubMed content"""
    # Try different patterns to find title
    patterns = [
        r'TI\s*-\s*(.+?)\n',  # Standard PubMed format
        r'<ArticleTitle>(.+?)</ArticleTitle>',  # XML format
        r'Title:\s*(.+?)\n'  # Alternative format
    ]
    
    for pattern in patterns:
        match = re.search(pattern, content)
        if match:
            return match.group(1).strip()
    return None

In [71]:
import pandas as pd
import openai
import time
import os
import re
import requests
import json
from tqdm import tqdm
from dotenv import load_dotenv
from fuzzywuzzy import fuzz
from urllib.parse import quote
import logging

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 1  # seconds
SIMILARITY_THRESHOLD = 75  # Fuzzy match threshold (%)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pubmed_processor.log'),
        logging.StreamHandler()
    ]
)

def load_reference_data(csv_path):
    """Load and preprocess the reference data"""
    try:
        df = pd.read_csv(csv_path, encoding='latin1')
        
        # Clean titles without removing essential punctuation
        df['clean_title'] = df['Publication_Title'].apply(
            lambda x: clean_title(x) if pd.notna(x) else ""
        )
        
        # Get unique papers
        unique_papers = df.drop_duplicates('clean_title', keep='first')
        logging.info(f"Loaded {len(df)} records, found {len(unique_papers)} unique papers")
        return unique_papers
    except Exception as e:
        logging.error(f"Failed to load data: {e}")
        return None

def clean_title(title):
    """Clean title while preserving meaningful punctuation"""
    if not title or not isinstance(title, str):
        return ""
    
    # Basic cleaning
    title = title.strip()
    title = re.sub(r'\s+', ' ', title)  # Remove extra spaces
    title = re.sub(r'\.\.\.+', '', title)  # Remove ellipses
    
    return title

def search_pubmed(title):
    """Comprehensive PubMed search with multiple fallback strategies"""
    if not title:
        return None
    
    strategies = [
        lambda: exact_match_search(title),
        lambda: partial_match_search(title),
        lambda: first_part_search(title, words=10),
        lambda: fuzzy_match_search(title)
    ]
    
    for strategy in strategies:
        pmid = strategy()
        if pmid:
            return pmid
    
    return None

def exact_match_search(title):
    """Search with exact title match"""
    params = {
        'db': 'pubmed',
        'term': quote(f'"{title}"[Title]'),
        'retmode': 'json',
        'retmax': 1,
        'sort': 'relevance'
    }
    return execute_search(params, "exact")

def partial_match_search(title):
    """Search without quotes for partial matches"""
    params = {
        'db': 'pubmed',
        'term': quote(f'{title}[Title]'),
        'retmode': 'json',
        'retmax': 3,
        'sort': 'relevance'
    }
    return execute_search(params, "partial")

def first_part_search(title, words=8):
    """Search using first part of long titles"""
    if len(title.split()) <= words:
        return None
        
    first_part = ' '.join(title.split()[:words])
    params = {
        'db': 'pubmed',
        'term': quote(f'"{first_part}"[Title]'),
        'retmode': 'json',
        'retmax': 3,
        'sort': 'relevance'
    }
    return execute_search(params, "first_part")

def fuzzy_match_search(title):
    """Fallback to fuzzy matching"""
    params = {
        'db': 'pubmed',
        'term': quote(f'{title}[Title]'),
        'retmode': 'json',
        'retmax': 5,
        'sort': 'relevance'
    }
    
    try:
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=params,
            timeout=20
        )
        response.raise_for_status()
        data = response.json()
        
        for pmid in data.get('esearchresult', {}).get('idlist', []):
            content = fetch_article(pmid)
            pubmed_title = extract_title_from_content(content)
            
            if pubmed_title and fuzz.ratio(title.lower(), pubmed_title.lower()) >= SIMILARITY_THRESHOLD:
                logging.info(f"Fuzzy match found ({fuzz.ratio(title.lower(), pubmed_title.lower())}%)")
                return pmid
    except Exception as e:
        logging.warning(f"Fuzzy search failed: {str(e)[:100]}")
    
    return None

def execute_search(params, strategy_name):
    """Execute PubMed search with given parameters"""
    try:
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=params,
            timeout=15
        )
        response.raise_for_status()
        data = response.json()
        
        if int(data.get('esearchresult', {}).get('count', 0)) > 0:
            logging.info(f"Found via {strategy_name} match")
            return data['esearchresult']['idlist'][0]
    except Exception as e:
        logging.warning(f"{strategy_name} search failed: {str(e)[:100]}")
    
    return None

def fetch_article(pmid):
    """Fetch full article content from PubMed"""
    try:
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params={
                'db': 'pubmed',
                'id': pmid,
                'retmode': 'text',
                'rettype': 'abstract'
            },
            timeout=15
        )
        response.raise_for_status()
        return response.text
    except Exception as e:
        logging.error(f"Failed to fetch PMID {pmid}: {e}")
        return None

def extract_title_from_content(content):
    """Extract title from PubMed content"""
    patterns = [
        r'TI\s*-\s*(.+?)\n',
        r'<ArticleTitle>(.+?)</ArticleTitle>',
        r'Title:\s*(.+?)\n'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, content)
        if match:
            return match.group(1).strip()
    return None

def analyze_paper(content, title):
    """Analyze paper content with GPT"""
    prompt = f"""
    Extract from this research paper:
    
    1. EXPOSURES (what was studied as potential causes):
    - List specific exposure terms
    - Include exact names of chemicals, pollutants, treatments
    
    2. OUTCOMES (measured health effects):
    - List specific outcome terms  
    - Include exact health measures, biomarkers
    
    Return JSON format:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}
    
    Paper Title: {title}
    Content: {content[:10000]}
    """
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=500,
                request_timeout=30
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            logging.warning(f"Analysis attempt {attempt+1} failed: {str(e)[:100]}")
            time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return None

def process_papers(papers_df, limit=None):
    """Main processing pipeline"""
    results = []
    
    for idx, paper in tqdm(papers_df.iterrows(), total=len(papers_df)):
        title = paper['Publication_Title']
        if not title or not isinstance(title, str):
            continue
            
        # PubMed search
        pmid = search_pubmed(title)
        if not pmid:
            logging.info(f"No match for: {title[:60]}...")
            continue
            
        # Fetch content
        content = fetch_article(pmid)
        if not content:
            continue
            
        # Analyze content
        analysis = analyze_paper(content, title)
        if not analysis:
            continue
            
        # Store results
        results.append({
            'original_title': title,
            'pmid': pmid,
            'exposures': analysis.get('exposures', []),
            'outcomes': analysis.get('outcomes', []),
            'reference_exposure': paper.get('Exposure_Layer3', ''),
            'reference_outcome': paper.get('Outcome_Layer3', '')
        })
        
        if limit and len(results) >= limit:
            break
            
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return pd.DataFrame(results)

def main():
    # Load data
    papers_df = load_reference_data("CohortNetwork_ES&T_SI_B_Main(Table S1).csv")
    if papers_df is None:
        return
    
    # Process papers
    results_df = process_papers(papers_df)  # Remove limit for full processing
    
    # Save results
    if not results_df.empty:
        output_file = f"results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M')}.csv"
        results_df.to_csv(output_file, index=False)
        logging.info(f"Saved results to {output_file}")
        print(f"\n✅ Success! Processed {len(results_df)} papers")
    else:
        logging.warning("No results generated")
        print("\n❌ No results generated")

if __name__ == "__main__":
    main()

100%|█████████████████████████████████████████| 122/122 [03:36<00:00,  1.77s/it]


❌ No results generated


In [88]:
import pandas as pd
import requests
import time
from tqdm import tqdm
import re
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='pubmed_finder.log'
)

def clean_title(title):
    """Minimal title cleaning - just fix encoding issues"""
    if pd.isna(title):
        return ""
    title = str(title)
    # Only fix problematic characters, don't modify punctuation
    title = title.replace('\x92', "'").replace('\x96', '-')
    return title.strip()

def search_pubmed(title):
    """Search method that mirrors manual PubMed search behavior"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    
    # First try the exact title as you would paste it
    params = {
        'db': 'pubmed',
        'term': title.replace('"', ' '),  # Remove quotes if present
        'field': 'title',
        'retmode': 'json',
        'retmax': 3,
        'sort': 'relevance'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=20)
        response.raise_for_status()
        data = response.json()
        
        idlist = data.get('esearchresult', {}).get('idlist', [])
        if idlist:
            # Return first match without complex verification
            return idlist[0]
            
    except Exception as e:
        logging.warning(f"Search failed for '{title[:50]}...': {str(e)[:100]}")
    
    return None

def fetch_article_info(pmid):
    """Get basic article info for verification"""
    try:
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={
                'db': 'pubmed',
                'id': pmid,
                'retmode': 'json'
            },
            timeout=15
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        logging.error(f"Failed to fetch info for PMID {pmid}: {e}")
        return None

def process_papers(papers_df):
    """Simplified processing that actually works"""
    results = []
    
    for _, paper in tqdm(papers_df.iterrows(), total=len(papers_df)):
        title = clean_title(paper['Publication_Title'])
        if not title:
            continue
            
        # Search PubMed - using simple direct method
        pmid = search_pubmed(title)
        if not pmid:
            logging.info(f"No match for: {title[:60]}...")
            continue
            
        # Get basic info to confirm
        article_info = fetch_article_info(pmid)
        if not article_info:
            continue
            
        # Store the successful match
        results.append({
            'original_title': title,
            'pmid': pmid,
            'pubmed_title': article_info.get('result', {}).get(str(pmid), {}).get('title', ''),
            'exposure': paper.get('Exposure_Layer3', ''),
            'outcome': paper.get('Outcome_Layer3', '')
        })
        
        # Brief pause to avoid rate limiting
        time.sleep(0.5)
    
    return pd.DataFrame(results)

def main():
    print("Starting PubMed matching...")
    
    # Load data
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        unique_papers = df.drop_duplicates('Publication_Title', keep='first')
        print(f" Loaded {len(df)} records, found {len(unique_papers)} unique papers")
    except Exception as e:
        print(f" Failed to load data: {e}")
        return
    
    # Process papers
    results_df = process_papers(unique_papers)
    
    # Save results
    if not results_df.empty:
        output_file = "pubmed_matches.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\n Found {len(results_df)} matches!")
        print(f"Results saved to {output_file}")
        
        # Show sample
        print("\nSample matches:")
        print(results_df[['original_title', 'pmid']].head(3))
    else:
        print("\n No matches found - check pubmed_finder.log")

if __name__ == "__main__":
    main()

Starting PubMed matching...
 Loaded 428 records, found 122 unique papers


100%|█████████████████████████████████████████| 122/122 [02:14<00:00,  1.10s/it]


 Found 117 matches!
Results saved to pubmed_matches.csv

Sample matches:
                                      original_title      pmid
0  short- and intermediate-term exposure to ambie...  34717175
1  associations between solar and geomagnetic act...  34537201
2  solar activity is associated with diastolic an...  34713707


In [83]:
import pandas as pd
import openai
import requests
import time
import json
from tqdm import tqdm
from dotenv import load_dotenv
import logging
import os

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 1

# Enhanced logging setup
logging.basicConfig(
    level=logging.DEBUG,  # Changed to DEBUG for more detailed logs
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('paper_analysis_detailed.log'),
        logging.StreamHandler()
    ]
)

def log_dataframe_columns(df, name):
    """Helper function to log dataframe columns"""
    logging.debug(f"{name} columns: {list(df.columns)}")

def find_column(df, possible_names, required=True):
    """Find the correct column name with better error handling"""
    for name in possible_names:
        if name in df.columns:
            return name
    if required:
        error_msg = f"None of {possible_names} found in DataFrame columns. Available columns: {list(df.columns)}"
        logging.error(error_msg)
        raise KeyError(error_msg)
    return None

def fetch_full_article(pmid):
    """Fetch complete article text with better logging"""
    try:
        logging.debug(f"Fetching article for PMID: {pmid}")
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params={
                'db': 'pubmed',
                'id': pmid,
                'retmode': 'text',
                'rettype': 'abstract'
            },
            timeout=30  # Increased timeout
        )
        response.raise_for_status()
        content = response.text
        logging.debug(f"Successfully fetched content for PMID {pmid} (length: {len(content)})")
        return content
    except Exception as e:
        logging.error(f"Failed to fetch PMID {pmid}: {str(e)}")
        return None

def extract_elements_with_gpt(content, title):
    """Extract exposures and outcomes with better error handling"""
    if not content or len(content) < 100:
        logging.warning(f"Insufficient content for title: {title[:50]}...")
        return None

    prompt = f"""
    Analyze this scientific paper and extract:

    EXPOSURES (independent variables studied):
    - List specific chemicals, pollutants, treatments
    - Be precise with names (e.g. "PM2.5" not just "air pollution")
    - Include measures like "daily average concentration"

    OUTCOMES (dependent variables measured):
    - List specific health effects, biomarkers
    - Include measurement methods if relevant
    - Be precise (e.g. "FEV1 lung function" not just "lung health")

    Return JSON format:
    {{
        "exposures": ["exact_term1", "exact_term2"],
        "outcomes": ["exact_term1", "exact_term2"]
    }}

    Paper Title: {title}
    Content: {content[:12000]} [truncated if needed]
    """

    for attempt in range(MAX_RETRIES):
        try:
            logging.debug(f"GPT analysis attempt {attempt + 1} for: {title[:50]}...")
            response = openai.ChatCompletion.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=1000,
                response_format={"type": "json_object"},
                request_timeout=60  # Increased timeout
            )
            result = json.loads(response.choices[0].message.content)
            logging.debug(f"Successfully extracted elements for: {title[:50]}...")
            return result
        except json.JSONDecodeError as e:
            logging.error(f"Failed to parse GPT response for {title[:50]}...: {str(e)}")
            time.sleep(DELAY_BETWEEN_REQUESTS * 2)  # Longer wait for JSON errors
        except Exception as e:
            logging.warning(f"GPT attempt {attempt + 1} failed for {title[:50]}...: {str(e)[:100]}")
            time.sleep(DELAY_BETWEEN_REQUESTS)
    return None

def process_papers_with_analysis(papers_df):
    """Full processing pipeline with detailed logging"""
    results = []
    
    # Log input dataframe structure
    log_dataframe_columns(papers_df, "Input")
    
    # Find correct column names
    title_col = find_column(papers_df, ['Publication_Title', 'Title', 'Paper Title', 'original_title'])
    pmid_col = find_column(papers_df, ['pmid', 'PMID', 'PubMed ID'])
    exp_col = find_column(papers_df, ['Exposure_Layer3', 'Exposure', 'Exposures'], required=False)
    out_col = find_column(papers_df, ['Outcome_Layer3', 'Outcome', 'Outcomes'], required=False)
    
    logging.info(f"Using columns - Title: {title_col}, PMID: {pmid_col}, Exposure: {exp_col}, Outcome: {out_col}")

    for idx, paper in tqdm(papers_df.iterrows(), total=len(papers_df)):
        try:
            title = paper[title_col]
            if not title or not isinstance(title, str):
                logging.warning(f"Empty/invalid title at index {idx}")
                continue
                
            pmid = paper[pmid_col]
            if not pmid or pd.isna(pmid):
                logging.warning(f"Empty/invalid PMID for title: {title[:50]}...")
                continue

            logging.info(f"\nProcessing {idx + 1}/{len(papers_df)}: {title[:60]}... (PMID: {pmid})")
            
            # Get PubMed content
            content = fetch_full_article(pmid)
            if not content:
                continue
                
            # Extract elements with GPT
            extracted = extract_elements_with_gpt(content, title)
            if not extracted:
                logging.warning(f"No elements extracted for: {title[:50]}...")
                continue
            
            # Prepare reference data
            ref_exposure = paper[exp_col] if exp_col and exp_col in paper and pd.notna(paper[exp_col]) else ""
            ref_outcome = paper[out_col] if out_col and out_col in paper and pd.notna(paper[out_col]) else ""
            
            # Compare with reference data
            comparison = {
                'exposure_matches': [],
                'outcome_matches': [],
                'new_exposures': extracted.get('exposures', []),
                'new_outcomes': extracted.get('outcomes', []),
                'missing_exposures': [],
                'missing_outcomes': []
            }

            # Build result record
            result = {
                'pmid': pmid,
                'original_title': title,
                'gpt_extracted_exposures': extracted.get('exposures', []),
                'gpt_extracted_outcomes': extracted.get('outcomes', []),
                'reference_exposures': ref_exposure,
                'reference_outcomes': ref_outcome,
                **comparison
            }
            
            results.append(result)
            logging.info(f"Successfully processed: {title[:50]}...")
            
        except Exception as e:
            logging.error(f"Error processing row {idx}: {str(e)}")
        
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    if results:
        return pd.DataFrame(results)
    else:
        logging.error("No results were generated from any papers")
        return pd.DataFrame()

def main():
    try:
        # Load your successful matches
        logging.info("Loading input CSV file...")
        matches_df = pd.read_csv("pubmed_matches.csv")
        log_dataframe_columns(matches_df, "Input CSV")
        
        # Process with GPT analysis
        logging.info("Starting paper analysis...")
        analysis_df = process_papers_with_analysis(matches_df)
        
        if not analysis_df.empty:
            # Save comprehensive results
            output_file = "full_analysis_results.csv"
            analysis_df.to_csv(output_file, index=False)
            logging.info(f"Saved {len(analysis_df)} papers to {output_file}")
            print(f"\n🎉 Analysis complete! Saved {len(analysis_df)} papers")
            
            # Generate comparison report
            report = "Analysis completed successfully. See CSV for detailed results."
            with open("analysis_report.txt", "w") as f:
                f.write(report)
            print(report)
        else:
            logging.error("No papers were successfully processed")
            print("\n❌ No papers were processed successfully - check paper_analysis_detailed.log for errors")
            
    except Exception as e:
        logging.error(f"Fatal error in main: {str(e)}")
        print(f"\n🔥 Critical error: {str(e)}")

if __name__ == "__main__":
    main()

100%|█████████████████████████████████████████| 118/118 [06:32<00:00,  3.33s/it]


❌ No papers were processed successfully - check paper_analysis_detailed.log for errors


In [92]:
import pandas as pd
import openai
import requests
import time
import json
from tqdm import tqdm
from dotenv import load_dotenv
import logging
import re

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MAX_RETRIES = 3
DELAY_BETWEEN_REQUESTS = 1

# Setup proper logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('paper_extraction.log', mode='w'),  # Overwrite old log
        logging.StreamHandler()
    ]
)

def load_excel_data():
    """Load original Excel data with exposures/outcomes"""
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        logging.info(f"Loaded Excel data with {len(df)} records")
        return df
    except Exception as e:
        logging.error(f"Failed to load Excel: {e}")
        return None

def search_pubmed(title):
    """Search PubMed directly for each paper"""
    try:
        params = {
            'db': 'pubmed',
            'term': f'"{title}"[Title]',
            'field': 'title',
            'retmode': 'json',
            'retmax': 1,
            'sort': 'relevance'
        }
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=params,
            timeout=20
        )
        data = response.json()
        if int(data.get('esearchresult', {}).get('count', 0)) > 0:
            return data['esearchresult']['idlist'][0]
    except Exception as e:
        logging.error(f"PubMed search failed for '{title[:50]}...': {e}")
    return None

def fetch_full_paper(pmid):
    """Get complete paper content from PubMed"""
    try:
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params={
                'db': 'pubmed',
                'id': pmid,
                'retmode': 'text',
                'rettype': 'abstract'
            },
            timeout=30
        )
        return response.text
    except Exception as e:
        logging.error(f"Failed to fetch PMID {pmid}: {e}")
        return None

def extract_with_gpt(content, title):
    """Extract exposures/outcomes from paper content"""
    prompt = f"""
    Extract from this research paper:

    1. EXPOSURES (what was studied as potential causes):
    - List 3-5 specific exposure terms
    - Example: "PM2.5 levels", "lead exposure"

    2. OUTCOMES (measured health effects):
    - List 3-5 specific outcome terms  
    - Example: "lung function", "cognitive scores"

    Return JSON format:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}

    Paper Title: {title}
    Content: {content[:10000]}
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=1000,
            request_timeout=45
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        logging.error(f"GPT extraction failed: {e}")
        return None

def process_papers():
    """Main processing workflow"""
    # Load original data
    excel_data = load_excel_data()
    if excel_data is None:
        return None

    results = []
    
    # Process each unique paper
    unique_papers = excel_data.drop_duplicates('Publication_Title')
    logging.info(f"Processing {len(unique_papers)} unique papers")
    
    for _, paper in tqdm(unique_papers.iterrows(), total=len(unique_papers)):
        title = paper['Publication_Title']
        if not title or not isinstance(title, str):
            continue

        # 1. Search PubMed
        pmid = search_pubmed(title)
        if not pmid:
            logging.info(f"No PubMed match for: {title[:60]}...")
            continue

        # 2. Fetch full paper
        content = fetch_full_paper(pmid)
        if not content:
            continue

        # 3. Extract with GPT
        extracted = extract_with_gpt(content, title)
        if not extracted:
            continue

        # 4. Compare with Excel data
        excel_exposures = str(paper['Exposure_Layer3']).split(';') if pd.notna(paper['Exposure_Layer3']) else []
        excel_outcomes = str(paper['Outcome_Layer3']).split(';') if pd.notna(paper['Outcome_Layer3']) else []

        # 5. Save results
        results.append({
            'original_title': title,
            'pmid': pmid,
            'gpt_exposures': extracted.get('exposures', []),
            'gpt_outcomes': extracted.get('outcomes', []),
            'excel_exposures': excel_exposures,
            'excel_outcomes': excel_outcomes,
            'content_length': len(content)
        })

        time.sleep(DELAY_BETWEEN_REQUESTS)

    return pd.DataFrame(results)

def main():
    logging.info("Starting fresh paper extraction...")
    
    results_df = process_papers()
    
    if results_df is not None and not results_df.empty:
        output_file = "fresh_extraction_results.csv"
        results_df.to_csv(output_file, index=False)
        logging.info(f"Saved {len(results_df)} results to {output_file}")
        print(f"\n🎉 Success! Processed {len(results_df)} papers")
        print(f"Results saved to {output_file}")
        
        # Show sample
        print("\nSample extractions:")
        print(results_df[['original_title', 'gpt_exposures', 'gpt_outcomes']].head(3))
    else:
        logging.error("No papers were processed successfully")
        print("\n❌ No results generated - check paper_extraction.log")

if __name__ == "__main__":
    main()

100%|█████████████████████████████████████████| 122/122 [00:31<00:00,  3.82it/s]


❌ No results generated - check paper_extraction.log


In [96]:
import pandas as pd
import requests
import time
import random
from tqdm import tqdm
import logging
from bs4 import BeautifulSoup

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='pubmed_direct_search.log',
    filemode='w'
)

# User-agent rotation to avoid blocking
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)',
    'Mozilla/5.0 (X11; Linux x86_64)'
]

def search_pubmed_direct(title):
    """Search PubMed directly through their website with proper headers"""
    try:
        headers = {
            'User-Agent': random.choice(USER_AGENTS),
            'Accept': 'text/html,application/xhtml+xml',
            'Accept-Language': 'en-US,en;q=0.5'
        }
        
        # First try exact search
        search_url = f"https://pubmed.ncbi.nlm.nih.gov/?term={requests.utils.quote(title)}&sort=relevance"
        response = requests.get(search_url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Parse HTML response
        soup = BeautifulSoup(response.text, 'html.parser')
        results = soup.find_all('a', class_='docsum-title')
        
        if results:
            # Extract first result's PMID
            first_result = results[0]
            pmid = first_result.get('href', '').split('/')[-2]
            if pmid.isdigit():
                return pmid
        
        # If no results, try with quotes
        search_url = f"https://pubmed.ncbi.nlm.nih.gov/?term=\"{requests.utils.quote(title)}\"&sort=relevance"
        response = requests.get(search_url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.text, 'html.parser')
        results = soup.find_all('a', class_='docsum-title')
        
        if results:
            first_result = results[0]
            pmid = first_result.get('href', '').split('/')[-2]
            if pmid.isdigit():
                return pmid
        
        return None
        
    except Exception as e:
        logging.error(f"Direct search failed for '{title[:50]}...': {str(e)[:100]}")
        return None

def fetch_article_direct(pmid):
    """Fetch article through PubMed website"""
    try:
        headers = {
            'User-Agent': random.choice(USER_AGENTS),
            'Accept': 'text/html,application/xhtml+xml'
        }
        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract abstract
        abstract = soup.find('div', class_='abstract-content')
        if abstract:
            return abstract.get_text(separator=' ', strip=True)
        
        # Fallback to abstract in meta
        meta_abstract = soup.find('meta', {'name': 'description'})
        if meta_abstract:
            return meta_abstract.get('content', '')
        
        return None
        
    except Exception as e:
        logging.error(f"Failed to fetch PMID {pmid}: {str(e)[:100]}")
        return None

def process_papers():
    """Main processing function"""
    try:
        df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
        unique_papers = df.drop_duplicates('Publication_Title')
        logging.info(f"Loaded {len(unique_papers)} unique papers")
        
        results = []
        
        for _, paper in tqdm(unique_papers.iterrows(), total=len(unique_papers)):
            title = str(paper['Publication_Title'])
            if not title.strip():
                continue
            
            # Search PubMed directly
            pmid = search_pubmed_direct(title)
            if not pmid:
                logging.info(f"No match for: {title[:60]}...")
                continue
            
            # Fetch article content
            content = fetch_article_direct(pmid)
            if not content:
                continue
            
            # Store results
            results.append({
                'original_title': title,
                'pmid': pmid,
                'exposure': paper.get('Exposure_Layer3', ''),
                'outcome': paper.get('Outcome_Layer3', ''),
                'content_length': len(content)
            })
            
            # Respectful delay
            time.sleep(random.uniform(1, 3))
        
        return pd.DataFrame(results)
    
    except Exception as e:
        logging.error(f"Processing failed: {str(e)}")
        return None

def main():
    print("Starting direct PubMed search...")
    results_df = process_papers()
    
    if results_df is not None and not results_df.empty:
        output_file = "direct_pubmed_results.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\n Success! Found {len(results_df)} papers")
        print(f"Results saved to {output_file}")
        
        # Show sample
        print("\nSample matches:")
        print(results_df[['original_title', 'pmid']].head(3))
    else:
        print("\n No matches found - check pubmed_direct_search.log")

if __name__ == "__main__":
    main()

Starting direct PubMed search...


100%|█████████████████████████████████████████| 122/122 [07:37<00:00,  3.75s/it]


🎉 Success! Found 122 papers
Results saved to direct_pubmed_results.csv

Sample matches:
                                      original_title      pmid
0  short- and intermediate-term exposure to ambie...  37550674
1                                                nan  37731213
2  associations between solar and geomagnetic act...  38648690


In [98]:
df1 = pd.read_csv("direct_pubmed_results.csv")
df1.head()

,original_title,pmid,exposure,outcome,content_length
0,short- and intermediate-term exposure to ambie...,37550674,short-term PM2.5 mass,genes-450K,1950
1,NaN,37731213,short-term PM2.5 elemental components,genes-450K,1311
2,associations between solar and geomagnetic act...,38648690,interplanetary magnetic field,total white blood cell,1469
3,solar activity is associated with diastolic an...,34713707,interplanetary magnetic field,diastolic blood pressure,1783
4,"short-term air pollution, cognitive performanc...",34841262,short-term PM2.5 mass,MMSE score,1130


In [100]:
import pandas as pd
import requests
import time
import random
from tqdm import tqdm
import logging
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import json

# Configuration
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MAX_RETRIES = 3
DELAY = random.uniform(1, 3)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='direct_extraction.log',
    filemode='w'
)

def search_pubmed(title):
    """Search PubMed directly through their website"""
    try:
        search_url = f"https://pubmed.ncbi.nlm.nih.gov/?term={requests.utils.quote(title)}"
        response = requests.get(
            search_url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=30
        )
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract first result's PMID
        result = soup.find('a', class_='docsum-title')
        if result:
            return result.get('href', '').split('/')[-2]
        return None
    except Exception as e:
        logging.error(f"Search failed for '{title[:50]}...': {e}")
        return None

def fetch_article(pmid):
    """Fetch full article text from PubMed"""
    try:
        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        response = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=30
        )
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Get abstract text
        abstract = soup.find('div', class_='abstract-content')
        if abstract:
            return abstract.get_text(separator=' ', strip=True)
        
        # Fallback to meta description
        meta = soup.find('meta', {'name': 'description'})
        if meta:
            return meta.get('content', '')
        
        return None
    except Exception as e:
        logging.error(f"Fetch failed for PMID {pmid}: {e}")
        return None

def extract_elements(content, title):
    """Use ChatGPT to extract exposures and outcomes"""
    prompt = f"""
    Extract from this research paper:

    EXPOSURES (what was studied as potential causes):
    - List 3-5 specific exposure terms
    - Example: "PM2.5 levels", "lead exposure"

    OUTCOMES (measured health effects):
    - List 3-5 specific outcome terms  
    - Example: "lung function", "cognitive scores"

    Return JSON format:
    {{
        "exposures": ["term1", "term2"],
        "outcomes": ["term1", "term2"]
    }}

    Paper Title: {title}
    Content: {content[:10000]}
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=1000,
            request_timeout=45
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        logging.error(f"Extraction failed: {e}")
        return None

def process_papers():
    """Main processing function"""
    # Load only titles from Excel
    df = pd.read_csv("CohortNetwork_ES&T_SI_B_Main(Table S1).csv", encoding='latin1')
    unique_titles = df['Publication_Title'].unique()
    
    results = []
    
    for title in tqdm(unique_titles):
        if not isinstance(title, str):
            continue
            
        # Search PubMed
        pmid = search_pubmed(title)
        if not pmid:
            continue
            
        # Fetch article
        content = fetch_article(pmid)
        if not content:
            continue
            
        # Extract elements
        elements = extract_elements(content, title)
        if not elements:
            continue
            
        # Store only extracted data
        results.append({
            'original_title': title,
            'pmid': pmid,
            'extracted_exposures': elements.get('exposures', []),
            'extracted_outcomes': elements.get('outcomes', []),
            'content_length': len(content)
        })
        
        time.sleep(DELAY)
    
    return pd.DataFrame(results)

def main():
    print("Starting direct extraction from PubMed...")
    results_df = process_papers()
    
    if not results_df.empty:
        output_file = "extracted_elements.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\n Success! Extracted data from {len(results_df)} papers")
        print(f"Results saved to {output_file}")
        
        # Show sample
        print("\nSample extractions:")
        print(results_df[['original_title', 'extracted_exposures', 'extracted_outcomes']].head(3))
    else:
        print("\n No extractions completed - check direct_extraction.log")

if __name__ == "__main__":
    main()

Starting direct extraction from PubMed...


100%|█████████████████████████████████████████| 122/122 [03:46<00:00,  1.86s/it]


❌ No extractions completed - check direct_extraction.log
